<p class="lead">This notebook expands the seventh unit of the course, in the R language the laboratories use. The lecture slides are a map; this text is the territory. Units 2–6 were about <em>learning</em>: given data, fit a model that predicts. This unit is about the other half of every learning system, the one that has been hidden inside the <code>fit</code> step since Unit 2 — <em>optimisation</em>, the search for the parameters, the feature subset, the architecture or the molecule that makes some objective as large as possible. Gradient descent, the optimiser of Units 2 and 6, needs a smooth objective and a derivative. A great many problems in bioengineering have neither: which features to keep, in which order to visit a set of points, which combination of beam angles delivers a radiation dose, which molecular graph binds a receptor. <strong>Genetic algorithms</strong> (GAs) attack such problems by imitating natural selection: a population of candidate solutions is evaluated, the better ones are selected, recombined and mutated, and the cycle repeats until good solutions emerge. The unit defines the method, dissects each of its components with code, traces Goldberg's classic example by hand, builds a reusable implementation and tests it on three benchmark problems, states the theory that is supposed to explain why it works (and the criticisms of that theory), places the GA among its relatives — evolution strategies, genetic programming, differential evolution, particle swarms, simulated annealing — and closes with the two applications you will meet most often in machine learning, feature selection and hyperparameter tuning, together with the biomedical applications documented in the literature.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Explain why optimisation is a pillar of AI distinct from learning, and name four properties of an objective function (discrete, non-differentiable, rugged, black-box) that defeat gradient methods.</li>
<li>Define a genetic algorithm as a population-based stochastic search guided by a fitness function, and map the biological vocabulary (gene, chromosome, genotype, phenotype, population, generation, fitness, selection, crossover, mutation) onto its algorithmic meaning.</li>
<li>Identify, in any published GA, its eight components — encoding, fitness, selection, crossover, mutation, replacement, termination and parameters — and describe at least two alternatives for each.</li>
<li>Trace one full generation of Goldberg's \(f(x) = x^2\) example by hand: fitness table, expected counts, roulette selection, one-point crossover and mutation.</li>
<li>Implement a complete GA in base R for binary, real-valued and permutation encodings and apply it to a multimodal continuous function, a knapsack problem and a travelling-salesperson instance; compare it with random search, differential evolution and gradient descent.</li>
<li>State the schema theorem and the building-block hypothesis, explain their intuition, summarise the main objections, and measure population diversity in code.</li>
<li>Describe evolution strategies and CMA-ES, genetic programming, differential evolution, particle swarm optimisation and simulated annealing, and state what the no-free-lunch theorem does and does not say.</li>
<li>Use a GA as a wrapper for feature selection and hyperparameter tuning, and explain why the fitness of the selected solution is an optimistically biased estimate that must be checked on a held-out test set (Supplement S1).</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Each code chunk is short and commented. Run it, read its output, then read the paragraph that interprets the output. Everything runs in a Google Colab R runtime; in a fresh runtime install the packages once with <code>install.packages(c("e1071","randomForest","rpart.plot","neuralnet","cluster","mlbench","pROC"))</code>. This unit needs only base R and the <code>mlbench</code>, <code>class</code>, <code>e1071</code> and <code>MASS</code> packages, so no data is downloaded. There is <strong>no genetic-algorithm package</strong>: the GA is written from scratch in base R, exactly as the Python edition wrote it from scratch — that is the point of the unit. Every random draw is seeded, so the numbers quoted are the ones you will see, but a GA is a stochastic algorithm, and the first thing to do after reading a result is to change the seed and see how much it moves. One algorithm is traced by hand (Section 4) before the code confirms it.</p></div>



In [ ]:
# Libraries used throughout. Random seeds are fixed so every number quoted is reproducible.
suppressPackageStartupMessages({
  library(mlbench); library(class); library(e1071); library(MASS)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#8a8a8a"
# Two helpers used everywhere: a bit-vector to a string, and binary (most significant bit first) to integer.
bits2str <- function(v) paste(v, collapse = "")
decode   <- function(P) { P <- rbind(P); L <- ncol(P); as.vector(P %*% 2^((L - 1):0)) }
cat(R.version.string, "| mlbench", as.character(packageVersion("mlbench")),
    "| e1071", as.character(packageVersion("e1071")), "| class", as.character(packageVersion("class")), "\n")



The printout records the R and package versions these notes were built with. The whole unit relies on base R and the four packages above;<sup class="c"><a href="#r36">[36]</a></sup><sup class="c"><a href="#r37">[37]</a></sup> the modelling functions (<code>knn</code>, <code>svm</code>) come from the standard R machine-learning stack.<sup class="c"><a href="#r38">[38]</a></sup>

## 1. Introduction: optimisation and search as the third pillar

### 1.1 Learning, inference and optimisation

Russell and Norvig open their textbook with <em>problem solving by search</em> before any chapter on learning, and for a reason: almost every method of this course is, underneath, a search. Logistic regression (Unit 2) searches for the weight vector that minimises the cross-entropy loss; a decision tree (Unit 4) searches greedily for the split that most reduces impurity; k-means (Unit 5) searches for the centroids that minimise within-cluster variance; a neural network (Unit 6) searches for its weights by gradient descent. The three pillars of an intelligent system are thus <strong>representation</strong> (how knowledge and data are encoded — Unit 1), <strong>inference and learning</strong> (how conclusions and models are drawn from data — Units 2–6) and <strong>optimisation and search</strong> (how the best candidate is found among a space of alternatives). Unit 1 listed four approaches to AI — symbolic, connectionist, statistical and evolutionary — and this unit is the evolutionary one: here the search itself, rather than the model, is the object of study.

### 1.2 When gradient methods fail

The optimisers of Units 2 and 6 follow the same recipe: compute the gradient of the objective with respect to the parameters, take a small step downhill, repeat. The recipe is extraordinarily effective when its assumptions hold and useless when they do not. Four properties of an objective break it:

- **Discrete search spaces.** A feature subset is a binary vector; a route through a set of points is a permutation; a treatment plan selects a set of beam angles. There is no gradient with respect to a bit, and there are no "small steps" between permutations.
- **Non-differentiable or piecewise-constant objectives.** The accuracy of a classifier as a function of its hyperparameters is a step function: changing the \(C\) of an SVM by a tiny amount usually changes no prediction, so the derivative is zero almost everywhere.
- **Rugged (multimodal) landscapes.** Even when the objective is smooth, gradient descent stops at the first local minimum it meets. On a landscape with hundreds of local optima the result depends entirely on the starting point.
- **Black-box objectives.** When the objective is the output of a simulation, a laboratory assay or a clinical score, there is no formula to differentiate; the only operation available is "evaluate this candidate".

The chunk below shows three objectives. The first is the convex bowl that gradient descent is designed for. The second is a rugged one-dimensional function (a slice of the Rastrigin function of Section 5); gradient descent from twelve random starts is run on it and the twelve end-points marked. The third is a discrete objective — the cross-validated accuracy of a classifier as a function of how many of the top-ranked features it uses — on which a gradient does not even apply.



In [ ]:
rastrigin_1d <- function(x) 10 + x^2 - 10 * cos(2 * pi * x)
xs <- seq(-5.12, 5.12, length.out = 600)
set.seed(0); starts <- runif(12, -5.12, 5.12)
ends <- sapply(starts, function(s)
  optim(s, function(v) rastrigin_1d(v), method = "L-BFGS-B", lower = -5.12, upper = 5.12)$par)

# Breast-cancer data (mlbench): the R stand-in for sklearn's load_breast_cancer. 9 nuclear features.
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]; bc <- bc[complete.cases(bc), ]
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))
X_all <- as.matrix(bc[, 1:9]); y_all <- bc$Class
set.seed(0); n <- nrow(X_all); tr_idx <- sample(n, round(0.7 * n))
Xc_tr <- X_all[tr_idx, ]; yc_tr <- y_all[tr_idx]; Xc_te <- X_all[-tr_idx, ]; yc_te <- y_all[-tr_idx]

cv_knn_acc <- function(X, y, folds = 5, kk = 5) {       # 5-fold CV accuracy of a k-NN, standardised inside each fold
  X <- as.matrix(X); set.seed(0); fid <- sample(rep(1:folds, length.out = nrow(X)))
  mean(sapply(1:folds, function(fo) {
    tr <- fid != fo; sc <- scale(X[tr, , drop = FALSE])
    ctr <- attr(sc, "scaled:center"); scl <- attr(sc, "scaled:scale"); scl[scl == 0] <- 1
    Xte <- scale(X[!tr, , drop = FALSE], center = ctr, scale = scl)
    mean(knn(sc, Xte, y[tr], k = kk) == y[!tr])
  }))
}
Fstat <- sapply(1:9, function(j) summary(aov(Xc_tr[, j] ~ yc_tr))[[1]][1, "F value"])
ord_feat <- order(Fstat, decreasing = TRUE)
acc_k <- sapply(1:9, function(k) cv_knn_acc(Xc_tr[, ord_feat[1:k], drop = FALSE], yc_tr))

par(mfrow = c(1, 3), mar = c(4, 4, 2.4, 1))
plot(xs, xs^2, type = "l", col = BLUE, lwd = 2, xlab = "x", ylab = "f(x)", main = "convex: gradient descent works")
plot(xs, rastrigin_1d(xs), type = "l", col = BLUE, lwd = 1, xlab = "x", ylab = "f(x)", main = "rugged: 12 starts, 12 answers")
points(ends, rastrigin_1d(ends), pch = 19, col = ORANGE, cex = 1.1)
plot(1:9, acc_k, type = "s", col = BLUE, lwd = 2, xlab = "number of top-ranked features kept",
     ylab = "5-fold CV accuracy (k-NN)", main = "discrete: no gradient exists")
cat(sprintf("gradient descent on the rugged function: %d of 12 runs reached the global minimum at x = 0\n",
            sum(abs(ends) < 1e-3)))



<p class="caption"><b>Figure 1.</b> Left: a convex objective, where every downhill path leads to the single minimum. Centre: a one-dimensional slice of the Rastrigin function; the orange points are where gradient descent (L-BFGS-B) stopped from twelve random starts — each run converged to the nearest local minimum, and the number that reached the global minimum at \(x = 0\) is printed above. Right: cross-validated accuracy of a k-NN classifier on the breast-cancer training data as a function of how many of the top-ranked features it uses; the objective is defined only at integers and is not monotone, so "take a step downhill" is meaningless.</p>

### 1.3 The biological metaphor and its limits

The idea of using simulated evolution to solve problems was developed independently several times in the 1960s — Rechenberg and Schwefel's <em>evolution strategies</em> in Berlin, Fogel's <em>evolutionary programming</em> in California — but the <em>genetic algorithm</em> in its canonical form, with binary chromosomes, fitness-proportional selection, crossover and mutation, is due to John Holland and his students at the University of Michigan. Holland's 1975 book framed it as a general theory of adaptive systems and supplied the <em>schema theorem</em> (Section 6);<sup class="c"><a href="#r1">[1]</a></sup> Kenneth De Jong's 1975 thesis was the first systematic experimental study of GA behaviour and established the methodology of parameter studies that Section 3.8 follows;<sup class="c"><a href="#r3">[3]</a></sup> and David Goldberg's 1989 textbook, which contains the hand-worked example of Section 4, made the method accessible to engineers.<sup class="c"><a href="#r2">[2]</a></sup>

The metaphor is productive but must not be taken literally, and the differences matter for the engineer:

| In biology | In a genetic algorithm |
|---|---|
| Fitness is <em>defined</em> by reproductive success; there is no external objective | Fitness is a function <em>chosen by the designer</em>; the algorithm optimises exactly what the function rewards, which is often not what was intended |
| The genotype-to-phenotype map (development) is enormously complex | The decoding function is usually simple and known |
| Populations are huge and time is not a constraint | Populations of 20–200 and a budget of a few thousand fitness evaluations are typical, because each evaluation may be a simulation or a cross-validation |
| Mutation and recombination are the only sources of novelty | The designer may add local search, repair operators, problem-specific heuristics |
| Evolution has no goal and does not converge | A GA is expected to converge, and <em>premature</em> convergence (Section 6) is its characteristic failure |

<div class="box warn"><h4>Common misconception</h4><p>A genetic algorithm is not a learning algorithm in the sense of Units 2–6: it does not produce a model from data, it produces the <em>best candidate</em> for an objective. When a GA is used in machine learning it is used <em>around</em> a learning algorithm — to choose its features, its hyperparameters or its architecture — and the learning algorithm is called, as a black box, inside the fitness function. Section 8 shows both uses. The distinction to keep is that the GA searches a space of <em>solutions</em>, not a space of <em>hypotheses about data</em>.</p></div>

### 1.4 Where this unit sits

Unit 1 named four approaches to AI. The symbolic, connectionist and statistical approaches have each had three or four units; the evolutionary approach has this one. Its methods are older than deep learning, slower than gradient descent when a gradient exists, and irreplaceable when one does not. In bioengineering they appear wherever a design space is combinatorial: the selection of electrodes or features from a physiological signal,<sup class="c"><a href="#r33">[33]</a></sup> the arrangement of beams in radiotherapy planning,<sup class="c"><a href="#r29">[29]</a></sup> the sequence of a protein with a prescribed fold,<sup class="c"><a href="#r32">[32]</a></sup> and the molecular graph of a drug candidate.<sup class="c"><a href="#r30">[30]</a></sup> They are also, since 2017, the main competitor of deep generative models in de-novo molecular design, a point Section 8 returns to.

## 2. Definition and vocabulary

### 2.1 Definition

A <strong>genetic algorithm</strong> is a <em>population-based, stochastic search method</em> that iteratively improves a set of candidate solutions by applying operators inspired by natural selection and genetics — selection, recombination (crossover) and mutation — using an objective function (the <em>fitness</em>) as the only information about the problem. Each clause does work:

- <em>Population-based</em>: unlike gradient descent or simulated annealing, which follow one point, a GA maintains many candidates at once. This is what lets it explore several regions of the search space simultaneously, and what makes crossover possible.
- <em>Stochastic</em>: selection, crossover and mutation are random. Two runs with different seeds give different results; the output of a GA is a distribution, and good practice reports the mean and spread over several runs.
- <em>Search</em>, not <em>learning</em>: the output is a candidate solution, not a model.
- <em>Fitness as the only information</em>: the algorithm needs no derivative, no formula, no continuity. It needs a function it can call. This is both its generality and its cost — a GA that needs 5,000 evaluations is cheap when an evaluation is a formula and prohibitive when it is a week-long experiment.

Formally, let \(\Omega\) be the search space (binary strings of length \(L\), real vectors in a box, permutations of \(n\) items) and \(f: \Omega \to \mathbb{R}\) the fitness, to be maximised. A GA maintains a population \(P_t = \{x_1, \dots, x_N\}\) at generation \(t\) and produces \(P_{t+1}\) by

<div class="eq">\[ P_{t+1} = \text{Replace}\big(P_t,\ \text{Mutate}\big(\text{Crossover}\big(\text{Select}(P_t, f)\big)\big)\big) \]
<span class="where">where Select draws \(N\) parents from \(P_t\) with probability increasing in \(f\), Crossover pairs them and exchanges parts of their chromosomes, Mutate perturbs the offspring at random, and Replace decides which of the old and new individuals survive. The cycle stops when a termination criterion (Section 3.7) is met, and the best individual ever seen is returned.</span></div>

### 2.2 Vocabulary: biology to algorithm

The authors of the GA literature use biological vocabulary, and each term has a precise algorithmic meaning:

| Biology | Algorithm | Example (feature selection, Section 8.1) |
|---|---|---|
| **Gene** | One position (locus) of the solution vector and the value (allele) it holds | bit 7, which is 1 if feature 7 is used |
| **Chromosome** (individual) | The complete encoded solution, a vector of genes | a binary string |
| **Genotype** | The encoded representation that the operators act on | `0110100` |
| **Phenotype** | The decoded solution that the fitness evaluates | the k-NN classifier trained on the chosen features |
| **Population** | A multiset of \(N\) chromosomes | several feature masks |
| **Generation** | One iteration of the loop | — |
| **Fitness** | \(f(\text{phenotype})\), the objective to be maximised | 5-fold cross-validated accuracy |
| **Selection** | Choosing parents with probability increasing in fitness | tournament of size 3 |
| **Crossover** | Producing a child from two parents by combining their genes | one-point crossover |
| **Mutation** | Random perturbation of a child's genes | flipping each bit with small probability |
| **Elitism** | Copying the best individual(s) unchanged into the next generation | keep the best 2 |

The distinction between <em>genotype</em> and <em>phenotype</em> is the one most often blurred. The operators act on genotypes: a one-point crossover of two bit strings is well defined whether or not the result decodes to a sensible classifier. The fitness is a property of phenotypes: it is the classifier, not the bit-string, whose accuracy is measured. The <strong>decoding</strong> function between them is where most of the design effort of a GA goes, because a bad encoding makes good solutions unreachable or the operators destructive (Section 3.1). The chunk below shows every term of the table on a population of four chromosomes.



In [ ]:
# Four 5-bit chromosomes encoding integers 0..31, fitness f(x) = x^2 (Goldberg's example, traced by hand in Section 4).
population <- rbind(c(0,1,1,0,1), c(1,1,0,0,0), c(0,1,0,0,0), c(1,0,0,1,1))
phenotype  <- decode(population)
fitness    <- phenotype^2
tab <- data.frame(
  `chromosome (genotype)` = apply(population, 1, bits2str),
  `x (phenotype)`         = phenotype,
  `fitness f(x)=x^2`      = as.integer(fitness),
  `selection prob.`       = round(fitness / sum(fitness), 3),
  `expected copies`       = round(fitness / mean(fitness), 2),
  check.names = FALSE)
cat(sprintf("population size N = %d, chromosome length L = %d, search space |Omega| = 2^L = %d\n",
            nrow(population), ncol(population), 2^ncol(population)))
cat(sprintf("sum of fitness = %.0f, mean = %.1f, max = %.0f\n", sum(fitness), mean(fitness), max(fitness)))
print(tab)



Four chromosomes, four phenotypes, four fitness values. The last two columns anticipate Section 3.3: under fitness-proportional selection the second chromosome, with almost half of the total fitness, expects almost two copies in the next generation and the third, with about 5 %, expects a fifth of a copy — in practice it will usually disappear. This is already the whole mechanism: good genes spread, poor genes vanish, and the operators of the next section recombine what survives.



In [ ]:
# The generational loop of a GA, drawn as a schematic with base graphics.
plot.new(); plot.window(xlim = c(0, 10), ylim = c(0, 3))
steps <- c("Initialise\npopulation","Evaluate\nfitness","Termination\nmet?","Select\nparents","Crossover","Mutate","Replace\n(+elitism)")
cols  <- c(BLUE, GREEN, GREY, ORANGE, ORANGE, ORANGE, BLUE)
xpos  <- c(0.3, 1.7, 3.1, 4.6, 6.0, 7.4, 8.8)
for (i in seq_along(steps)) {
  rect(xpos[i], 1.1, xpos[i] + 1.15, 2.0, col = cols[i], border = NA)
  text(xpos[i] + 0.575, 1.55, steps[i], col = "white", font = 2, cex = 0.72)
}
for (i in 1:(length(xpos) - 1)) arrows(xpos[i] + 1.15, 1.55, xpos[i + 1], 1.55, length = 0.07, lwd = 1)
segments(9.95, 1.1, 9.95, 0.5); segments(9.95, 0.5, 2.85, 0.5)
arrows(2.85, 0.5, 2.85, 1.1, length = 0.07, lwd = 1)
text(6.0, 0.62, "next generation (t = t + 1)", cex = 0.8, font = 3)
arrows(3.675, 2.0, 3.675, 2.7, length = 0.07, lwd = 1)
text(5.1, 2.75, "yes: return best individual ever seen", cex = 0.8)
text(4.0, 1.02, "no", cex = 0.8)
title(main = "One generation of a genetic algorithm")



<p class="caption"><b>Figure 2.</b> The generational loop. Everything in orange is where the "genetics" happens — selection, crossover, mutation — and is independent of the problem once the encoding is fixed; everything in green depends on the problem and nothing else. This separation is the practical appeal of the method: the same loop solves a feature-selection problem, a routing problem and a dose-optimisation problem, with only the decoding and the fitness rewritten.</p>

## 3. The elements of a genetic algorithm

Every GA you will read about is a choice at each of eight points: (1) how solutions are encoded, (2) how fitness is computed and scaled, (3) how parents are selected, (4) how they are recombined, (5) how offspring are mutated, (6) which individuals survive, (7) when to stop, and (8) the numerical parameters. This section takes them in turn, each with a code chunk, using the terminology of Eiben and Smith's textbook, the modern standard reference.<sup class="c"><a href="#r5">[5]</a></sup>

### 3.1 Encoding and decoding

The encoding decides what a chromosome <em>is</em>. Four families cover almost every application:

- **Binary.** The canonical GA uses strings of bits. Anything can be binary-encoded: an integer by its base-2 representation, a real number \(x \in [a, b]\) by an \(L\)-bit integer \(k\) with \(x = a + k\,(b-a)/(2^L - 1)\) — so resolution is \((b-a)/(2^L-1)\) and more bits mean finer resolution — and a subset by a mask. The weakness of plain binary coding of numbers is the <em>Hamming cliff</em>: 7 = `0111` and 8 = `1000` are neighbours in value but differ in all four bits, so a mutation that moves 7 to 8 is very unlikely. <em>Gray codes</em>, in which consecutive integers differ in exactly one bit, remove the cliff.
- **Integer.** When the genes are naturally integers with a meaning (the number of neurons in a layer, a beam angle in degrees, a categorical choice), encoding them directly avoids the artificiality of bits.
- **Real-valued.** For continuous parameters, a chromosome is simply a vector \(\mathbf{x} \in \mathbb{R}^d\); the operators change accordingly, giving a <em>real-coded GA</em>, which is close to an evolution strategy.<sup class="c"><a href="#r9">[9]</a></sup>
- **Permutation.** When the solution is an ordering — a route, a schedule — a chromosome is a permutation. Standard crossover would produce repeated and missing items, so specialised operators (order crossover, Section 3.4) are required.<sup class="c"><a href="#r10">[10]</a></sup>

The chunk below encodes and decodes one example of each.



In [ ]:
encode_real <- function(x, a, b, L) {        # real x in [a,b] -> L-bit plain-binary chromosome
  k <- round((x - a) / (b - a) * (2^L - 1)); as.integer(intToBits(k))[L:1]
}
decode_real <- function(chrom, a, b) { L <- length(chrom); a + decode(chrom) * (b - a) / (2^L - 1) }
to_gray <- function(bits) c(bits[1], bitwXor(bits[-1], bits[-length(bits)]))   # g_i = b_i XOR b_{i-1}

a <- -5.12; b <- 5.12; L <- 10
ex <- data.frame(
  encoding = c("binary (real x = 1.5, 10 bits)", "binary, Gray of 7 and 8",
               "integer (neurons per layer)", "real-valued (log10 C, log10 gamma)",
               "permutation (route through 6 cities)"),
  `chromosome (genotype)` = c(
     bits2str(encode_real(1.5, a, b, L)),
     paste(bits2str(to_gray(c(0,1,1,1))), "/", bits2str(to_gray(c(1,0,0,0)))),
     "64, 16, 4", "1.2, -2.7", "3, 0, 5, 1, 4, 2"),
  `decoded solution (phenotype)` = c(
     sprintf("x = %.4f (resolution %.4f)", decode_real(encode_real(1.5, a, b, L), a, b), (b - a)/(2^L - 1)),
     "plain 0111/1000 differ in 4 bits; Gray codes differ in 1",
     "layers of 64, 16 and 4 units",
     sprintf("C = %.1f, gamma = %.4f", 10^1.2, 10^-2.7),
     "visit city 3, then 0, 5, 1, 4, 2, return to 3"),
  check.names = FALSE)
print(ex)



The first row shows the price of binary coding of a real number: with 10 bits over \([-5.12, 5.12]\) the resolution is about 0.01, and the value 1.5 is represented by the nearest grid point, not exactly. Every extra bit halves the resolution but doubles the search space. The second row shows the Hamming cliff and its Gray-code cure. The remaining rows need no decoding beyond interpretation, which is the argument for using the natural representation of the problem whenever one exists.<sup class="c"><a href="#r5">[5]</a></sup>

### 3.2 The fitness function and fitness scaling

The fitness function is the only channel through which the problem speaks to the algorithm, so three rules govern its design. First, <em>it must be a maximisation</em>; a cost \(c(x)\) to be minimised becomes a fitness \(f(x) = -c(x)\) or \(1/(1 + c(x))\) (the latter keeps fitness positive, which fitness-proportional selection requires). Second, <em>it must reflect what is wanted</em>, including constraints: an infeasible solution is either repaired, rejected, or <em>penalised</em> by subtracting a term proportional to the violation; Section 5.2 uses a penalty. Third, <em>its scale matters</em> for fitness-proportional selection, because selection probabilities are ratios of fitness values: adding a constant to every fitness changes the probabilities although it changes nothing about the problem.

The third point produces two opposite failures that Goldberg's book treats at length.<sup class="c"><a href="#r2">[2]</a></sup> Early in a run, one individual may be far fitter than the rest — a <em>super-individual</em> — and take over the population before anything else is explored. Late in a run, when every individual is nearly as good as every other, fitness-proportional selection degenerates to random selection and progress stalls. Two remedies are standard:

- **Linear scaling** replaces \(f\) by \(f' = \alpha f + \beta\), with \(\alpha, \beta\) chosen each generation so that the mean is preserved and the best individual receives a fixed number of expected copies, \(f'_{\max} = C\, f_{\text{avg}}\), typically \(C = 2\) (negative scaled values clipped to zero).
- **Rank-based fitness** discards the fitness values and uses only their ranks: the worst receives selection probability proportional to \(2 - s\) and the best to \(s\), with \(1 \le s \le 2\), interpolating linearly.<sup class="c"><a href="#r8">[8]</a></sup> The selection pressure is then constant throughout the run.

The chunk below applies the three schemes to the two problematic situations.



In [ ]:
p_proportional <- function(f) f / sum(f)
linear_scaling <- function(f, C = 2.0) {           # Goldberg: mean preserved, max = C * mean
  fa <- mean(f); fmax <- max(f); fmin <- min(f)
  if (fmin > (C * fa - fmax) / (C - 1)) {
    alpha <- (C - 1) * fa / (fmax - fa); beta <- fa * (fmax - C * fa) / (fmax - fa)
  } else { alpha <- fa / (fa - fmin); beta <- -fmin * fa / (fa - fmin) }
  pmax(alpha * f + beta, 0)
}
p_rank <- function(f, s = 1.8) {                   # linear ranking: probability depends on rank only
  N <- length(f); ranks <- rank(f, ties.method = "first") - 1
  (2 - s) / N + 2 * ranks * (s - 1) / (N * (N - 1))
}
early <- c(980, 40, 35, 30, 25, 20)                # a super-individual dominates
late  <- c(101, 100.5, 100, 99.8, 99.5, 99)        # everyone nearly equal
par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
for (cs in list(list(early, "early run: one super-individual"), list(late, "late run: nearly equal fitness"))) {
  f <- cs[[1]]
  M <- rbind(raw = p_proportional(f), scaled = p_proportional(linear_scaling(f)), rank = p_rank(f))
  barplot(M, beside = TRUE, col = c(BLUE, ORANGE, GREEN), names.arg = sprintf("f=%g", f),
          cex.names = 0.7, ylab = "selection probability", main = cs[[2]], ylim = c(0, 1))
}
legend("topright", c("raw proportional", "linear scaling C=2", "linear ranking s=1.8"),
       fill = c(BLUE, ORANGE, GREEN), bty = "n", cex = 0.8)
for (nm in list(list("early", early), list("late", late))) {
  f <- nm[[2]]
  cat(sprintf("%s: expected copies of best  raw %.2f | scaled %.2f | rank %.2f\n", nm[[1]],
      length(f) * max(p_proportional(f)), length(f) * max(p_proportional(linear_scaling(f))), length(f) * max(p_rank(f))))
}



<p class="caption"><b>Figure 3.</b> Selection probabilities of six individuals under three fitness transformations. Left: with raw proportional selection the super-individual (fitness 980) takes most of the probability and expects several copies in a population of six — the rest is almost certain to vanish in one generation. Right: when every fitness is within 2 % of the others, raw proportional selection gives everyone one sixth and the search becomes a random walk. Linear scaling and ranking produce the same moderate pressure — the best expects about two copies — in both situations (the expected copies are printed above).</p>

### 3.3 Selection

Selection creates the pressure towards better solutions. Its strength, the <strong>selection pressure</strong>, is the central tuning knob: too weak and the population drifts, too strong and it collapses onto the current best before better regions are found. Four schemes account for nearly every published GA.

**Roulette-wheel (fitness-proportional) selection** is Holland's original. Each individual receives a slice of a wheel proportional to its fitness, \(p_i = f_i / \sum_j f_j\), and the wheel is spun \(N\) times. The expected number of copies of individual \(i\) is \(N p_i = f_i / \bar f\). Its two problems were described in Section 3.2; a third is <em>sampling error</em>: with only \(N\) spins the actual counts can be far from the expected ones.

**Stochastic universal sampling (SUS)**, due to Baker, spins the wheel once with \(N\) equally spaced pointers instead of \(N\) times with one pointer, so sampling error is minimised while expected values are unchanged.<sup class="c"><a href="#r8">[8]</a></sup>

**Tournament selection** picks \(k\) individuals at random and returns the best. It needs no fitness scaling, works with negative or cost-type fitness, and its pressure is set by one integer: \(k = 2\) is mild, \(k = 7\) is strong. It is the most common choice in modern practice.<sup class="c"><a href="#r8">[8]</a></sup>

**Rank selection** applies roulette or SUS to the rank-based fitness of Section 3.2, so pressure is constant and independent of the fitness scale.

Blickle and Thiele introduced a clean way to measure selection pressure: the <strong>takeover time</strong>, the number of generations that selection alone (no crossover, no mutation) needs to fill the population with copies of the single best individual.<sup class="c"><a href="#r8">[8]</a></sup> The first chunk draws the roulette wheel; the second implements the four schemes and measures their takeover times.



In [ ]:
labs <- sprintf("%s (x=%d, f=%d)", apply(population, 1, bits2str), phenotype, as.integer(fitness))
pie(fitness, labels = labs, col = c(BLUE, ORANGE, GREEN, GREY),
    main = "Roulette wheel: P(select i) = f_i / sum f", cex = 0.85)



<p class="caption"><b>Figure 4.</b> The roulette wheel of Goldberg's initial population. Chromosome 11000 occupies about half the wheel and 01000 about 5 %. Spinning the wheel four times gives the mating pool; the expected counts are 0.58, 1.97, 0.22 and 1.23 copies (Section 2.2), but a particular set of four spins can give any counts at all — Section 4 traces one such set by hand.</p>



In [ ]:
select_roulette  <- function(f, n) sample.int(length(f), n, replace = TRUE, prob = f / sum(f))
select_sus <- function(f, n) {                       # one spin, n equally spaced pointers
  cum <- cumsum(f / sum(f)); ptr <- (runif(1) + 0:(n - 1)) / n
  pmin(findInterval(ptr, cum) + 1, length(f))
}
select_tournament <- function(f, n, k = 2) {
  cand <- matrix(sample.int(length(f), n * k, replace = TRUE), ncol = k)
  cand[cbind(1:n, max.col(matrix(f[cand], ncol = k), ties.method = "first"))]
}
select_rank <- function(f, n, s = 1.8) sample.int(length(f), n, replace = TRUE, prob = p_rank(f, s))

takeover_time <- function(selector, N = 80, n_runs = 20, max_gen = 300) {
  times <- integer(n_runs); won <- logical(n_runs)
  for (r in 1:n_runs) {
    set.seed(r); f <- runif(N, 1, 2); f[1] <- 2.2; ids <- 1:N; g <- 0
    for (g in 1:max_gen) { ids <- ids[selector(f[ids], N)]; if (length(unique(ids)) == 1) break }
    times[r] <- g; won[r] <- (ids[1] == 1)
  }
  c(mean = mean(times), sd = sd(times), won = mean(won))
}
schemes <- list("roulette (proportional)" = function(f, n) select_roulette(f, n),
                "SUS (proportional)"      = function(f, n) select_sus(f, n),
                "tournament k=2" = function(f, n) select_tournament(f, n, 2),
                "tournament k=4" = function(f, n) select_tournament(f, n, 4),
                "tournament k=7" = function(f, n) select_tournament(f, n, 7),
                "linear rank s=1.8" = function(f, n) select_rank(f, n))
res <- t(sapply(schemes, takeover_time))
print(data.frame(scheme = rownames(res), `mean takeover (gen)` = round(res[, "mean"], 2),
           sd = round(res[, "sd"], 2), `fraction won by best` = round(res[, "won"], 2),
           check.names = FALSE, row.names = NULL))



The table orders the schemes by pressure. With fitness spread uniformly between 1 and 2 and a best individual only 10 % above the rest, proportional selection — roulette or SUS — needs many generations to make the population uniform, because its pressure depends on the fitness <em>ratios</em>, which here are small. The last column exposes a second defect of the roulette wheel: in a fraction of the runs the population converges to an individual <em>other than the best</em>, because the single copy of the best was lost by sampling error in an early spin (genetic drift); SUS loses it far less often, since its expected count is always rounded to at least one. Tournament selection takes over quickly, and raising \(k\) shortens the takeover time further. Rank selection with \(s = 1.8\) sits near tournament selection with \(k = 2\), as the theory predicts.<sup class="c"><a href="#r8">[8]</a></sup> None of these numbers is "right": the correct pressure depends on how much exploration the problem needs (Section 6).

<div class="box key"><h4>Exercise 1</h4><p>In <code>takeover_time</code>, change the fitness of the best individual from 2.2 to 20 (a super-individual) and re-run. Proportional selection now takes over almost immediately and never loses the best, while tournament and rank selection are unchanged. Explain, from the formulas, why the two families react so differently to the fitness scale.</p></div>

### 3.4 Crossover (recombination)

Crossover is the operator that distinguishes a GA from every other stochastic search method: it creates a child by <em>combining</em> parts of two parents, on the hypothesis that two good parents may hold different good parts and that a child inheriting both will be better than either (the <em>building-block hypothesis</em>, Section 6). It is applied with probability \(p_c\), typically 0.6–0.9. The operator depends on the encoding.

**Binary and integer chromosomes.** <em>One-point</em> crossover chooses a cut site \(c\) and swaps the tails. It preserves adjacent genes — a hidden bias (<em>positional bias</em>) if related genes are far apart on the string. <em>Two-point</em> crossover swaps the segment between two cut sites. <em>Uniform</em> crossover decides each gene independently by a coin flip; it has no positional bias and is the most disruptive.

**Real-valued chromosomes.** Swapping coordinates creates nothing new; real-coded operators generate children <em>between and around</em> the parents. <em>Arithmetic</em> crossover takes \(\mathbf{c} = \lambda \mathbf{a} + (1 - \lambda) \mathbf{b}\), so the population can only contract. <em>BLX-\(\alpha\)</em> (blend crossover) draws each child coordinate uniformly from \([\min - \alpha d,\ \max + \alpha d]\) with \(d = |a_i - b_i|\); with \(\alpha = 0.5\) the population can also expand.<sup class="c"><a href="#r9">[9]</a></sup> <em>Simulated binary crossover (SBX)</em> mimics the spread one-point crossover produces on binary-coded numbers, with a distribution index \(\eta\) controlling the spread; it is the default in NSGA-II.<sup class="c"><a href="#r22">[22]</a></sup>

**Permutations.** Any of the above would produce repeated and missing items. <em>Order crossover (OX)</em>, due to Davis, copies a random segment of parent \(a\) and fills the rest with the items of parent \(b\) in \(b\)'s order, skipping items already present. It preserves adjacencies from both parents.<sup class="c"><a href="#r10">[10]</a></sup>

The first chunk shows the three binary operators; the second draws the offspring distribution of the three real-valued operators; the third traces order crossover.



In [ ]:
one_point <- function(a, b, site = NULL) {
  L <- length(a); if (is.null(site)) site <- sample(1:(L - 1), 1)
  list(c(a[1:site], b[(site + 1):L]), c(b[1:site], a[(site + 1):L]))
}
two_point <- function(a, b, sites = NULL) {
  L <- length(a); if (is.null(sites)) sites <- sort(sample(1:(L - 1), 2))
  c1 <- sites[1]; c2 <- sites[2]; ca <- a; cb <- b
  ca[(c1 + 1):c2] <- b[(c1 + 1):c2]; cb[(c1 + 1):c2] <- a[(c1 + 1):c2]; list(ca, cb)
}
uniform_x <- function(a, b, mask = NULL) {
  if (is.null(mask)) mask <- runif(length(a)) < 0.5
  list(ifelse(mask, a, b), ifelse(mask, b, a))
}
set.seed(0); A <- rep(1, 10); B <- rep(0, 10)        # all-ones and all-zeros parents make inheritance visible
cat(sprintf("parent a = %s   parent b = %s\n", bits2str(A), bits2str(B)))
op1 <- one_point(A, B, 4); op2 <- two_point(A, B, c(3, 7)); op3 <- uniform_x(A, B)
cat(sprintf("one-point, site 4  -> child1 = %s   child2 = %s\n", bits2str(op1[[1]]), bits2str(op1[[2]])))
cat(sprintf("two-point, sites 3,7 -> child1 = %s   child2 = %s\n", bits2str(op2[[1]]), bits2str(op2[[2]])))
cat(sprintf("uniform, random mask -> child1 = %s   child2 = %s\n", bits2str(op3[[1]]), bits2str(op3[[2]])))



The children are exactly what the definitions predict: one-point at site 4 copies the first four genes of one parent and the tail of the other; two-point swaps only the middle segment; uniform mixes gene by gene. On these all-ones and all-zeros parents every inherited bit is visible.



In [ ]:
blx_alpha   <- function(a, b, alpha = 0.5) { lo <- pmin(a, b); hi <- pmax(a, b); d <- hi - lo; runif(length(a), lo - alpha * d, hi + alpha * d) }
arithmetic_x <- function(a, b) { lam <- runif(1); lam * a + (1 - lam) * b }
sbx <- function(a, b, eta = 2) {
  u <- runif(length(a))
  beta <- ifelse(u <= 0.5, (2 * u)^(1 / (eta + 1)), (1 / (2 - 2 * u))^(1 / (eta + 1)))
  sgn <- ifelse(runif(length(a)) < 0.5, 1, -1); 0.5 * ((a + b) + sgn * beta * (b - a))
}
pa <- c(1, 1); pb <- c(3, 2); set.seed(0)
par(mfrow = c(1, 3), mar = c(4, 4, 2.4, 1))
for (op in list(list(arithmetic_x, "arithmetic (on the segment)"),
                list(function(a, b) blx_alpha(a, b), "BLX-0.5 (box, +50%)"),
                list(function(a, b) sbx(a, b), "SBX, eta=2 (near parents)"))) {
  kids <- t(sapply(1:400, function(i) op[[1]](pa, pb)))
  plot(kids, pch = 19, col = adjustcolor(BLUE, 0.4), cex = 0.5, xlim = c(-0.5, 4.5), ylim = c(-0.5, 3.5),
       xlab = "x1", ylab = "x2", main = op[[2]])
  points(rbind(pa, pb), pch = 19, col = ORANGE, cex = 1.6)
}



<p class="caption"><b>Figure 5.</b> Four hundred children of the same two parents (orange) under three real-valued crossover operators. Arithmetic crossover places every child on the segment between the parents, so a population can only shrink. BLX-0.5 fills a box that extends the parents' bounding box by half its width, so children can lie outside and the population can grow. SBX concentrates each child coordinate near one parent or the other, with a tail that reaches beyond them. The choice decides whether crossover is a contracting or expanding force, which sets the exploration–exploitation balance of Section 6.</p>



In [ ]:
order_crossover <- function(a, b, segment = NULL, verbose = FALSE) {
  n <- length(a)
  if (is.null(segment)) { ij <- sort(sample(0:(n - 1), 2)); i <- ij[1]; j <- ij[2] } else { i <- segment[1]; j <- segment[2] }
  child <- rep(-1, n); child[(i + 1):(j + 1)] <- a[(i + 1):(j + 1)]      # 1. copy segment of a (0-indexed i..j)
  b_order <- b[(((j + 1) + 0:(n - 1)) %% n) + 1]                         # 2. b read from after the segment, wrapping
  fill <- b_order[!(b_order %in% child)]                                 #    skipping items already present
  num <- n - (j - i + 1)                                                 #    positions to fill
  if (num > 0) { positions <- (((j + 1) + 0:(num - 1)) %% n) + 1; child[positions] <- fill }   # 3. wrapping
  if (verbose) {
    partial <- ifelse(seq_len(n) - 1 >= i & seq_len(n) - 1 <= j, as.character(a), "_")
    cat(sprintf("parent a = %s\nparent b = %s\nstep 1: positions %d..%d copied from a -> %s\n",
                paste(a, collapse = " "), paste(b, collapse = " "), i, j, paste(partial, collapse = " ")))
    cat(sprintf("step 2: b after segment, skipping present: %s\nstep 3: child = %s\n",
                paste(fill, collapse = " "), paste(child, collapse = " ")))
  }
  child
}
set.seed(0)
pa <- c(0,1,2,3,4,5,6,7); pb <- c(7,3,0,6,1,5,2,4)
child <- order_crossover(pa, pb, segment = c(2, 4), verbose = TRUE)
cat(sprintf("valid permutation: %s\n", identical(sort(child), 0:7)))



The child keeps cities 2, 3, 4 at positions 2–4 exactly as in parent \(a\), and fills the other positions with the remaining cities in the order parent \(b\) visits them, starting after the segment. Every city appears exactly once — the output is a valid route — and the child inherits <em>adjacencies</em> from both parents, which is what a crossover for routing problems should preserve, since the length of a route depends only on which cities are adjacent.<sup class="c"><a href="#r10">[10]</a></sup>

### 3.5 Mutation and the mutation rate

Mutation introduces genes the population does not already contain. Without it, a GA can only recombine the alleles present in the initial population; once an allele is lost from every chromosome, crossover cannot recreate it. Mutation is therefore the guarantee of <em>reachability</em> and the basis of the convergence proofs of Section 6.<sup class="c"><a href="#r12">[12]</a></sup>

- **Bit-flip** mutation (binary): each bit is inverted independently with probability \(p_m\); the standard recommendation is \(p_m \approx 1/L\), one flip per chromosome on average.<sup class="c"><a href="#r11">[11]</a></sup>
- **Gaussian** mutation (real-valued): \(x_i \leftarrow x_i + \mathcal{N}(0, \sigma^2)\). The step size \(\sigma\) plays the role of \(p_m\); evolution strategies (Section 7) adapt it during the run.
- **Swap** mutation (permutations): exchange the items at two random positions; the result is always valid. Variants are insert, inversion and scramble.

Mutation is <em>disruptive</em>: on average it makes a child worse. Its benefit is statistical — a small fraction of mutations are improvements, and selection keeps those — so the rate must be high enough to find them and low enough not to destroy what selection has built. The chunk below implements the three operators and a compact binary GA that the experiments of Sections 3.5–3.8 use, then measures the effect of \(p_m\) on the <em>OneMax</em> problem (maximise the number of ones in a 50-bit string, whose optimum is known).



In [ ]:
mutate_bitflip  <- function(kids, pm) { m <- matrix(runif(length(kids)) < pm, nrow = nrow(kids)); kids[m] <- 1 - kids[m]; kids }
mutate_gaussian <- function(kids, pm, sigma, lo = -Inf, hi = Inf) {
  m <- matrix(runif(length(kids)) < pm, nrow = nrow(kids))
  pmin(pmax(kids + m * matrix(rnorm(length(kids), 0, sigma), nrow = nrow(kids)), lo), hi)
}
mutate_swap <- function(kids, pm) {
  for (i in 1:nrow(kids)) if (runif(1) < pm) { ij <- sample(ncol(kids), 2); kids[i, ij] <- kids[i, rev(ij)] }
  kids
}
binary_ga <- function(fitness, L, n_pop = 50, n_gen = 60, pc = 0.8, pm = NULL, n_elite = 1, k = 2, stall = NULL) {
  if (is.null(pm)) pm <- 1 / L
  pop <- matrix(sample(0:1, n_pop * L, replace = TRUE), nrow = n_pop); fit <- fitness(pop)
  best <- pop[which.max(fit), ]; bestf <- max(fit); hist <- matrix(NA, 0, 2); since <- 0
  for (g in 1:n_gen) {
    hist <- rbind(hist, c(max(fit), mean(fit)))
    elite <- pop[order(fit, decreasing = TRUE)[seq_len(n_elite)], , drop = FALSE]
    par <- pop[select_tournament(fit, 2 * (n_pop - n_elite), k), , drop = FALSE]
    np <- nrow(par) / 2; kids <- matrix(0, np, L)
    for (i in 1:np) { a <- par[2 * i - 1, ]; b <- par[2 * i, ]; kids[i, ] <- if (runif(1) < pc) one_point(a, b)[[1]] else a }
    pop <- if (n_elite > 0) rbind(elite, mutate_bitflip(kids, pm)) else mutate_bitflip(kids, pm)
    fit <- fitness(pop); since <- if (max(fit) > bestf) 0 else since + 1
    if (max(fit) > bestf) { best <- pop[which.max(fit), ]; bestf <- max(fit) }
    if (!is.null(stall) && since >= stall) break
  }
  hist <- rbind(hist, c(max(fit), mean(fit)))
  list(best = best, best_fitness = bestf, history = hist)
}
onemax <- function(P) rowSums(P); L <- 50
plot(NULL, xlim = c(0, 81), ylim = c(25, 51), xlab = "generation", ylab = "best fitness (mean of 8 runs)",
     main = "OneMax, L=50, N=40: the mutation rate decides whether the GA finishes")
cols <- c(GREY, BLUE, GREEN, ORANGE); leg <- c()
for (j in seq_along(list(0, 1/L, 5/L, 0.5))) {
  pm <- c(0, 1/L, 5/L, 0.5)[j]
  runs <- sapply(1:8, function(r) { set.seed(r); binary_ga(onemax, L, n_pop = 40, n_gen = 80, pm = pm)$history[, 1] })
  lines(0:80, rowMeans(runs), col = cols[j], lwd = 2)
  leg <- c(leg, sprintf("p_m=%.2f (%.0f flips); final %.1f", pm, L * pm, mean(runs[nrow(runs), ])))
}
abline(h = L, lty = 3); legend("bottomright", leg, col = cols, lwd = 2, bty = "n", cex = 0.72)



<p class="caption"><b>Figure 6.</b> Best fitness per generation on OneMax (\(L = 50\), optimum 50) for four mutation rates, averaged over eight runs. Without mutation (grey) the population converges to whatever alleles it started with and stops short of the optimum. With \(p_m = 1/L\) (blue) the GA reaches, or nearly reaches, the optimum. With five expected flips per chromosome (green) progress is slower, and with \(p_m = 0.5\) (orange) every child is a random string and the GA is a random search. The final best values are in the legend.</p>

<div class="box key"><h4>Key idea</h4><p>Crossover <em>exploits</em>: it combines material that selection has already approved. Mutation <em>explores</em>: it supplies material that is not there. A GA with crossover and no mutation converges to a sub-space and stays there; a GA with mutation and no crossover is a population of parallel hill-climbers; the two together, at the right rates, are what the method is.</p></div>

### 3.6 Replacement and elitism

Replacement decides which individuals form the next generation. In the <strong>generational</strong> model, the \(N\) offspring replace the \(N\) parents entirely (Holland's original, traced in Section 4). In the <strong>steady-state</strong> model, one or two offspring are produced per step and replace the worst member, so generations overlap. Between the two lies the \((\mu + \lambda)\) scheme of evolution strategies (Section 7).

The pure generational model has a well-known defect: the best individual of generation \(t\) may fail to be selected, or be destroyed by crossover or mutation, so the best fitness can <em>decrease</em>. <strong>Elitism</strong> — copying the best \(e\) individuals unchanged into the next generation — removes the defect at negligible cost, and it is a condition of the convergence theorem of Section 6: Rudolph proved that the canonical GA <em>without</em> elitism does not converge to the optimum, whereas the elitist version does.<sup class="c"><a href="#r12">[12]</a></sup> The chunk below runs the same GA with and without elitism on the 2-D Rastrigin function, encoded with 12 bits per coordinate, and plots the best fitness <em>in the current population</em>.



In [ ]:
rastrigin  <- function(X) { X <- rbind(X); 10 * ncol(X) + rowSums(X^2 - 10 * cos(2 * pi * X)) }
decode_pop <- function(P, bits = 12, lo = -5.12, hi = 5.12) {
  P <- rbind(P); w <- 2^((bits - 1):0)
  k <- cbind(P[, 1:bits, drop = FALSE] %*% w, P[, (bits + 1):(2 * bits), drop = FALSE] %*% w)
  lo + k * (hi - lo) / (2^bits - 1)
}
fit_rastrigin_bin <- function(P) -rastrigin(decode_pop(P))
L2 <- 24
plot(NULL, xlim = c(0, 60), ylim = c(1e-3, 60), log = "y", xlab = "generation",
     ylab = "cost of best individual (log)", main = "2-D Rastrigin, binary-coded (12 bits/coordinate)")
for (e in list(list(0, ORANGE, "no elitism"), list(1, BLUE, "elitism (best 1 copied)"))) {
  set.seed(3); r <- binary_ga(fit_rastrigin_bin, L2, n_pop = 40, n_gen = 60, pm = 1 / L2, n_elite = e[[1]])
  lines(0:(nrow(r$history) - 1), -r$history[, 1], col = e[[2]], lwd = 2)
  drops <- sum(diff(r$history[, 1]) < -1e-12)
  cat(sprintf("%-26s: best cost in final pop %.4f; generations where best fitness fell: %d\n", e[[3]], -r$history[nrow(r$history), 1], drops))
}
legend("topright", c("no elitism", "elitism (best 1 copied)"), col = c(ORANGE, BLUE), lwd = 2, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 7.</b> Cost (negative fitness) of the best individual in the current population, on a logarithmic axis. Without elitism (orange) the curve rises repeatedly — the printed count is the number of generations in which the best individual was lost — because the best chromosome was not selected or was broken by crossover or mutation, and the run has to rediscover it. With elitism (blue) the curve is monotone by construction and the final cost is lower. The lesson: always keep the best solution found, whether by elitism in the population or by a separate "best ever" variable.</p>

### 3.7 Termination criteria

A GA has no natural stopping point. Four criteria are used, often combined with "whichever comes first":

1. **A fixed number of generations** (or of fitness evaluations, the fairer budget when comparing algorithms with different population sizes) — the only one that makes runtime predictable.
2. **A target fitness** has been reached — possible only when the optimum, or an acceptable value, is known in advance.
3. **Stagnation**: the best fitness has not improved by more than a tolerance in the last \(s\) generations. This adapts runtime to the problem but can stop a run that would have escaped a plateau later.
4. **Loss of diversity**: the population has become nearly uniform (Section 6), so further generations are mostly repetitions.

The chunk below adds a stagnation criterion to the previous experiment and reports where the run stops.



In [ ]:
for (stall in c(10, 15, 25)) {
  set.seed(0); r <- binary_ga(fit_rastrigin_bin, L2, n_pop = 40, n_gen = 300, pm = 1 / L2, n_elite = 1, stall = stall)
  cat(sprintf("stall = %2d generations -> stopped after %3d generations; best cost %.4f at x = (%s)\n",
      stall, nrow(r$history) - 1, -r$best_fitness, paste(round(decode_pop(matrix(r$best, nrow = 1))[1, ], 3), collapse = ", ")))
}



With this seed the stagnation windows stop a few generations after the last improvement, and longer windows only add generations. That is the usual outcome but not guaranteed: with another seed a short window may stop during a plateau that would have been left later, which is why stagnation criteria are combined with a minimum number of generations. Because binary coding with 12 bits per coordinate has a resolution of about 0.0025, the best reachable cost on this encoding is not exactly zero even when the GA finds the correct cell of the grid.

### 3.8 The parameters and their effects

A canonical GA has four numerical parameters — population size \(N\), crossover probability \(p_c\), mutation probability \(p_m\), and the selection pressure — plus the elitism count and the termination budget. De Jong's thesis recommended \(N = 50\), \(p_c = 0.6\), \(p_m = 0.001\);<sup class="c"><a href="#r3">[3]</a></sup> Grefenstette later found \(N = 30\), \(p_c = 0.95\), \(p_m = 0.01\) better for online performance; the practical advice that survived is \(p_m \approx 1/L\), \(p_c \in [0.6, 0.9]\), and \(N\) as large as the evaluation budget allows.<sup class="c"><a href="#r5">[5]</a></sup> Srinivas and Patnaik proposed making \(p_c\) and \(p_m\) <em>adaptive</em> — low for high-fitness individuals, high for low-fitness ones — which reduces premature convergence without hand tuning.<sup class="c"><a href="#r11">[11]</a></sup>

The interaction that matters most is between \(N\) and \(p_m\) <em>at a fixed evaluation budget</em>: a larger population explores more per generation but affords fewer generations. The chunk below fixes the budget at 4,000 fitness evaluations and measures the final cost on the binary-coded Rastrigin problem for a grid of \(N\) and \(p_m\), averaged over five seeds.



In [ ]:
budget <- 4000; sizes <- c(10, 20, 50, 100); rates <- c(0, 0.5 / L2, 1 / L2, 2 / L2, 5 / L2)
grid <- matrix(0, length(sizes), length(rates))
for (i in seq_along(sizes)) for (j in seq_along(rates)) {
  costs <- sapply(1:5, function(r) { set.seed(r)
    -binary_ga(fit_rastrigin_bin, L2, n_pop = sizes[i], n_gen = budget %/% sizes[i], pm = rates[j], n_elite = 1)$best_fitness })
  grid[i, j] <- mean(costs)
}
image(seq_along(rates), seq_along(sizes), t(log10(grid + 1e-3)), col = hcl.colors(25, "YlGnBu", rev = TRUE),
      axes = FALSE, xlab = "mutation rate p_m (L = 24)", ylab = "population size N",
      main = "Mean final cost (2-D Rastrigin, 4000 evals, 5 runs)")
axis(1, seq_along(rates), c("0", "0.5/L", "1/L", "2/L", "5/L")); axis(2, seq_along(sizes), sizes)
for (i in seq_along(sizes)) for (j in seq_along(rates)) text(j, i, sprintf("%.2f", grid[i, j]), cex = 0.8)
best_ij <- which(grid == min(grid), arr.ind = TRUE)[1, ]
cat(sprintf("best combination: N = %d, p_m = %.3f (%.1f flips/chromosome) -> mean cost %.3f\n",
    sizes[best_ij[1]], rates[best_ij[2]], rates[best_ij[2]] * L2, min(grid)))



<p class="caption"><b>Figure 8.</b> Mean final cost (lower is better; the numbers are the values and the colour its logarithm) of the binary-coded Rastrigin GA at a fixed budget of 4,000 evaluations. The column without mutation is poor at every population size, because the search is confined to the alleles of the initial population. The columns at \(p_m = 1/L\) and \(2/L\) are generally best; \(5/L\) is worse. The largest population runs out of generations; a very small population behaves like a mutation-driven hill-climber, which suits this smooth-at-small-scale landscape. Two cautions: the pattern is specific to this problem and encoding, and five runs give only a rough mean — re-run with more seeds and some cells will exchange ranks.</p>

<div class="box key"><h4>Exercise 2</h4><p>Change the budget to 1,000 and to 16,000 evaluations and re-run. How does the best population size move with the budget? Then replace the tournament size <code>k=2</code> by <code>k=5</code> inside <code>binary_ga</code> and observe whether stronger selection helps the large or the small populations more.</p></div>

## 4. Worked example by hand: Goldberg's x² problem

Goldberg's textbook opens with a hand simulation that every student of the field has reproduced.<sup class="c"><a href="#r2">[2]</a></sup> The problem is to maximise \(f(x) = x^2\) over the integers \(x \in \{0, 1, \dots, 31\}\). The encoding is a 5-bit binary string, so \(x = 31\) = `11111` is the optimum with \(f = 961\). The population has \(N = 4\), selection is roulette, crossover is one-point with \(p_c = 1\), and mutation is bit-flip with \(p_m = 0.001\). The initial population is the one Goldberg obtained by tossing a coin: 01101, 11000, 01000, 10011.

### 4.1 Step 1 — evaluate the population

| No. | string | \(x\) | \(f(x) = x^2\) | \(p_i = f_i / \sum f\) | expected copies \(f_i / \bar f\) |
|---|---|---|---|---|---|
| 1 | 01101 | 13 | 169 | 0.14 | 0.58 |
| 2 | 11000 | 24 | 576 | 0.49 | 1.97 |
| 3 | 01000 | 8 | 64 | 0.06 | 0.22 |
| 4 | 10011 | 19 | 361 | 0.31 | 1.23 |
| | **sum** | | **1170** | 1.00 | 4.00 |
| | **average** | | **293** | | |
| | **max** | | **576** | | |

### 4.2 Step 2 — roulette-wheel selection

The wheel has cumulative boundaries at \(0.144\), \(0.637\), \(0.692\) and \(1.000\). To make the example reproducible we take the four spins \(u = 0.10, 0.30, 0.55, 0.80\). Each \(u\) selects the first individual whose cumulative boundary exceeds it: \(0.10 \to\) No. 1; \(0.30 \to\) No. 2; \(0.55 \to\) No. 2; \(0.80 \to\) No. 4. The actual counts are therefore 1, 2, 0, 1 — close to the expected 0.58, 1.97, 0.22, 1.23, with the weakest individual eliminated. The mating pool is 01101, 11000, 11000, 10011.

### 4.3 Step 3 — one-point crossover

Following Goldberg, pool position 1 mates with 2 and position 3 with 4. The cut sites are **site 4** for the first pair and **site 2** for the second:

| pair | parents (cut marked) | children | \(x\) | \(f(x)\) |
|---|---|---|---|---|
| 1 × 2, site 4 | 0110\|1 and 1100\|0 | 01100 | 12 | 144 |
| | | 11001 | 25 | 625 |
| 3 × 4, site 2 | 11\|000 and 10\|011 | 11011 | 27 | 729 |
| | | 10000 | 16 | 256 |
| | | **sum** | | **1754** |
| | | **average** | | **439** |
| | | **max** | | **729** |

### 4.4 Step 4 — mutation

There are \(4 \times 5 = 20\) bits and each flips with probability \(0.001\), so the expected number of mutations is \(0.02\): the probability that at least one bit flips is \(1 - 0.999^{20} \approx 0.02\), and in this generation, as in Goldberg's, no bit flips. The new population is 01100, 11001, 11011, 10000.

### 4.5 Reading the result

In one generation the average fitness rose from 293 to 439 and the maximum from 576 to 729, and the population now contains 11011, a string that was not present before and that no individual could have produced alone: it combines the prefix `11` of the second parent with the suffix `011` of the fourth. That is the building-block argument of Section 6 in miniature. The schema `1****`, whose members all have \(x \ge 16\), went from two of four to three of four: above-average schemata gain representation, which is what the schema theorem predicts. The chunk below reproduces every number of the four tables with code, feeding the algorithm the same random numbers used by hand.



In [ ]:
pop0 <- rbind(c(0,1,1,0,1), c(1,1,0,0,0), c(0,1,0,0,0), c(1,0,0,1,1))
f_x2 <- function(P) decode(P)^2
f0 <- f_x2(pop0); psel <- f0 / sum(f0)
step1 <- data.frame(string = apply(pop0, 1, bits2str), x = decode(pop0), `f=x^2` = as.integer(f0),
                    p_select = round(psel, 3), `expected copies` = round(f0 / mean(f0), 2), check.names = FALSE)
cat("Step 1 - evaluation:\n"); print(step1, row.names = FALSE)
cat(sprintf("sum %.0f, average %.1f, max %.0f\n\n", sum(f0), mean(f0), max(f0)))

u <- c(0.10, 0.30, 0.55, 0.80)                          # the four roulette spins used by hand
chosen <- findInterval(u, cumsum(psel)) + 1
cat(sprintf("Step 2 - roulette: boundaries %s, spins %s -> individuals %s; counts %s\n",
    paste(round(cumsum(psel), 3), collapse = " "), paste(u, collapse = " "),
    paste(chosen, collapse = " "), paste(tabulate(chosen, 4), collapse = " ")))
pool <- pop0[chosen, ]

c12 <- one_point(pool[1, ], pool[2, ], site = 4); c34 <- one_point(pool[3, ], pool[4, ], site = 2)
pop1 <- rbind(c12[[1]], c12[[2]], c34[[1]], c34[[2]]); f1 <- f_x2(pop1)
step3 <- data.frame(pair = c("1x2,site4","1x2,site4","3x4,site2","3x4,site2"),
                    child = apply(pop1, 1, bits2str), x = decode(pop1), `f=x^2` = as.integer(f1), check.names = FALSE)
cat("\nStep 3 - one-point crossover:\n"); print(step3, row.names = FALSE)
cat(sprintf("sum %.0f, average %.1f, max %.0f\n", sum(f1), mean(f1), max(f1)))
set.seed(0); pm <- 0.001; flips <- matrix(runif(length(pop1)) < pm, nrow = nrow(pop1))
cat(sprintf("\nStep 4 - mutation: expected flips %.2f, P(>=1) = %.3f, flips drawn: %d\n",
    length(pop1) * pm, 1 - (1 - pm)^length(pop1), sum(flips)))
cat(sprintf("schema 1**** : %d of 4 before, %d of 4 after\n", sum(pop0[, 1] == 1), sum(pop1[, 1] == 1)))



Every figure matches the hand tables: sum 1170 to 1754, average 293 to 439, maximum 576 to 729, the children 01100, 11001, 11011 and 10000, no mutation, and the schema `1****` growing from two to three. Having confirmed one generation, the next chunk lets the same algorithm — now with random spins and sites — run for fifty generations with Goldberg's \(p_m = 0.001\) and, for comparison, \(p_m = 0.05\). With \(N = 4\) a run is at the mercy of a handful of random numbers, so ten seeds are shown for each.



In [ ]:
goldberg_ga <- function(pop, n_gen, pm = 0.001) {
  hist <- matrix(NA, 0, 2)
  for (g in 1:n_gen) {
    f <- f_x2(pop); hist <- rbind(hist, c(max(f), mean(f)))
    pool <- pop[select_roulette(f, nrow(pop)), , drop = FALSE]; pool <- pool[sample(nrow(pool)), , drop = FALSE]
    kids <- matrix(0, 0, ncol(pop))
    for (i in seq(1, nrow(pool), 2)) { ch <- one_point(pool[i, ], pool[i + 1, ]); kids <- rbind(kids, ch[[1]], ch[[2]]) }
    pop <- mutate_bitflip(kids, pm)
  }
  f <- f_x2(pop); hist <- rbind(hist, c(max(f), mean(f))); list(pop = pop, history = hist)
}
par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
for (pc in list(list(0.001, BLUE), list(0.05, ORANGE))) {
  pm <- pc[[1]]; plot(NULL, xlim = c(0, 50), ylim = c(0, 1000), xlab = "generation",
       ylab = "best fitness in population", main = sprintf("p_m = %s  (10 seeds, N = 4)", pm))
  ever <- finals <- integer(10)
  for (sd in 0:9) { set.seed(sd); r <- goldberg_ga(pop0, 50, pm = pm)
    lines(0:50, r$history[, 1], col = adjustcolor(pc[[2]], 0.45))
    finals[sd + 1] <- max(decode(r$pop)); ever[sd + 1] <- round(sqrt(max(r$history[, 1]))) }
  abline(h = 961, lty = 3)
  cat(sprintf("p_m = %s: x=31 found at some generation in %d of 10 runs; still present at the end in %d of 10\n",
      pm, sum(ever == 31), sum(finals == 31)))
}



<p class="caption"><b>Figure 9.</b> Best fitness per generation over ten runs of Goldberg's four-individual GA, with Goldberg's \(p_m = 0.001\) (left) and \(p_m = 0.05\) (right); the dotted line is the optimum \(f(31) = 961\). With \(p_m = 0.001\) every run converges within a few generations to a population of identical strings and stays there, and the optimum is found in few runs: with four individuals and almost no mutation, a lost allele is never recovered. With \(p_m = 0.05\) the optimum is reached at some generation far more often, but without elitism the roulette wheel or a mutation can discard it again (the printed summary gives both counts). The two panels show premature convergence without mutation and loss of the best without elitism — the reasons real applications use populations of tens to hundreds, \(p_m \approx 1/L\) and elitism.</p>

<div class="box warn"><h4>Where the textbook example simplifies</h4><p>Goldberg's example uses \(p_c = 1\) and no elitism, and its population of four is far too small for the roulette wheel to behave like its expected values — which is why its actual counts (1, 2, 0, 1) are illustrative rather than typical. In the modern formulation (Eiben and Smith; Whitley's tutorial) a GA uses \(N \ge 20\), tournament or rank selection, \(p_c \in [0.6, 0.9]\), \(p_m \approx 1/L\), elitism of one or two, and reports results over multiple seeds.<sup class="c"><a href="#r5">[5]</a>,<a href="#r7">[7]</a></sup> The arithmetic of the example is nonetheless exactly the arithmetic of every GA; only the parameters change.</p></div>

## 5. A reusable implementation, tested on three problems

Sections 3 and 4 used special-purpose code. This section writes a general GA whose only inputs are a fitness function and three encoding-specific operators (initialise, crossover, mutate). It is then tested on three problems spanning the three encodings: a continuous function with many local optima (real-valued), the 0/1 knapsack (binary) and a travelling-salesperson instance (permutation). For each, the GA is compared with <em>random search</em> at the same number of fitness evaluations — the minimum honest baseline for any stochastic optimiser: a method that does not beat random search at equal budget is not doing anything.



In [ ]:
# A reusable GA: the generational loop. Encoding-specific behaviour is passed in as functions.
run_ga <- function(fitness, init, crossover, mutate, n_pop = 50, n_gen = 100, pc = 0.9, k = 3, n_elite = 2, snapshots = c()) {
  pop <- init(n_pop); fit <- fitness(pop); n_eval <- n_pop
  best <- pop[which.max(fit), ]; bestf <- max(fit); hist <- matrix(NA, 0, 2); snaps <- list()
  for (g in 0:(n_gen - 1)) {
    hist <- rbind(hist, c(max(fit), mean(fit)))
    if (g %in% snapshots) snaps[[as.character(g)]] <- pop
    ord <- order(fit, decreasing = TRUE); elite <- pop[ord[seq_len(n_elite)], , drop = FALSE]
    par <- pop[select_tournament(fit, 2 * (n_pop - n_elite), k), , drop = FALSE]
    np <- nrow(par) / 2; kids <- matrix(0, np, ncol(pop))
    for (i in 1:np) { a <- par[2 * i - 1, ]; b <- par[2 * i, ]; kids[i, ] <- if (runif(1) < pc) crossover(a, b) else a }
    pop <- rbind(elite, mutate(kids)); fit <- fitness(pop); n_eval <- n_eval + n_pop
    if (max(fit) > bestf) { best <- pop[which.max(fit), ]; bestf <- max(fit) }
  }
  hist <- rbind(hist, c(max(fit), mean(fit)))
  if (n_gen %in% snapshots) snaps[[as.character(n_gen)]] <- pop
  list(best = best, best_fitness = bestf, history = hist, n_eval = n_eval, snapshots = snaps)
}
# Operator sets for the three encodings: each returns list(init, crossover, mutate).
ops_real <- function(lo, hi, d, pm = 0.2, sigma = 0.3, alpha = 0.5) list(
  init = function(n) matrix(runif(n * d, lo, hi), nrow = n),
  crossover = function(a, b) pmin(pmax(blx_alpha(a, b, alpha), lo), hi),
  mutate = function(kids) mutate_gaussian(kids, pm, sigma, lo, hi))
ops_binary <- function(L, pm = NULL) { if (is.null(pm)) pm <- 1 / L; list(
  init = function(n) matrix(sample(0:1, n * L, replace = TRUE), nrow = n),
  crossover = function(a, b) one_point(a, b)[[1]], mutate = function(kids) mutate_bitflip(kids, pm)) }
ops_permutation <- function(n_items, pm = 0.3) list(
  init = function(n) t(sapply(1:n, function(i) sample(0:(n_items - 1)))),
  crossover = function(a, b) order_crossover(a, b), mutate = function(kids) mutate_swap(kids, pm))
set.seed(0); res <- do.call(run_ga, c(list(onemax), ops_binary(50), list(n_pop = 40, n_gen = 60)))
cat(sprintf("smoke test on OneMax (L=50): best fitness %.0f of 50 after %d evaluations\n", res$best_fitness, res$n_eval))



The smoke test confirms the general loop solves OneMax exactly like the special-purpose `binary_ga` of Section 3.5. From here, only the three operator functions and the fitness change between problems.

### 5.1 A continuous multimodal benchmark: the Rastrigin function

The Rastrigin function, popularised as a GA benchmark by Mühlenbein and colleagues, is the standard test of an optimiser's ability to escape local minima:<sup class="c"><a href="#r14">[14]</a></sup>

<div class="eq">\[ f(\mathbf{x}) = 10\,d + \sum_{i=1}^{d} \left[x_i^2 - 10 \cos(2\pi x_i)\right], \qquad x_i \in [-5.12, 5.12] \]
<span class="where">where \(d\) is the dimension. The quadratic term makes the global minimum \(f(\mathbf{0}) = 0\); the cosine term adds a local minimum near every integer point, so in two dimensions there are about \(11^2 = 121\) local minima on a grid, each separated from its neighbours by a ridge of height about 20.</span></div>

Gradient descent from a random start falls into the nearest of the 121 minima (Figure 1). A GA with a population spread over the domain sees many basins at once, and selection concentrates it into the deepest. The chunk below runs the real-coded GA and photographs the population at four generations.



In [ ]:
lo <- -5.12; hi <- 5.12
set.seed(0); res_r <- do.call(run_ga, c(list(function(P) -rastrigin(P)), ops_real(lo, hi, 2),
                                        list(n_pop = 60, n_gen = 80, snapshots = c(0, 5, 15, 80))))
gx <- seq(lo, hi, length.out = 120); gy <- gx
Z <- outer(gx, gy, function(a, b) mapply(function(u, v) rastrigin(matrix(c(u, v), nrow = 1)), a, b))
par(mfrow = c(1, 4), mar = c(3.4, 3.4, 2.2, 0.6))
for (g in c(0, 5, 15, 80)) {
  image(gx, gy, Z, col = grey.colors(25, start = 0.1, end = 0.95), xlab = "x1", ylab = "x2")
  P <- res_r$snapshots[[as.character(g)]]
  points(P, pch = 21, bg = ORANGE, col = "black", cex = 0.7)
  points(0, 0, pch = 3, col = "red", lwd = 2, cex = 1.3)
  title(main = sprintf("gen %d: best cost %.3g", g, min(rastrigin(P))), cex.main = 0.95)
}
cat(sprintf("GA: best cost %.2e at x = (%s) after %d evaluations\n",
    rastrigin(matrix(res_r$best, nrow = 1)), paste(round(res_r$best, 4), collapse = ", "), res_r$n_eval))



<p class="caption"><b>Figure 10.</b> The 2-D Rastrigin landscape (dark = low cost; the red cross is the global minimum) with the population of 60 at generations 0, 5, 15 and 80. The initial population is uniform over the square. By generation 5 selection has emptied the high-cost regions; by generation 15 it has concentrated in the central basin, with a few individuals still probing neighbours (mutation and BLX crossover keep producing them); by generation 80 the population sits on the optimum to within the Gaussian mutation width. The printed final cost is far below the ridge height, so the GA has found the global basin, not merely a good local one.</p>

The next chunk makes the comparison quantitative, over ten seeds at an equal budget, against pure random search; a <em>differential-evolution</em> algorithm (Storn and Price's method, implemented from scratch here and described in Section 7);<sup class="c"><a href="#r18">[18]</a></sup> and gradient descent (L-BFGS-B) restarted from random points until the budget is used up — the honest way to use a local optimiser on a multimodal problem.



In [ ]:
# Differential evolution (rand/1/bin) from scratch, and the comparison. cost1 maps a vector to a scalar cost.
differential_evolution <- function(cost1, lo, hi, d, NP = 20, Fw = 0.7, CR = 0.9, maxiter = 100) {
  pop <- matrix(runif(NP * d, lo, hi), nrow = NP); fit <- apply(pop, 1, cost1); nev <- NP
  for (it in 1:maxiter) for (i in 1:NP) {
    idx <- sample.int(NP, 3); while (i %in% idx) idx <- sample.int(NP, 3)
    v <- pmin(pmax(pop[idx[1], ] + Fw * (pop[idx[2], ] - pop[idx[3], ]), lo), hi)
    cr <- runif(d) < CR; cr[sample(d, 1)] <- TRUE
    trial <- ifelse(cr, v, pop[i, ]); ft <- cost1(trial); nev <- nev + 1
    if (ft < fit[i]) { pop[i, ] <- trial; fit[i] <- ft }
  }
  list(x = pop[which.min(fit), ], fun = min(fit), nfev = nev)
}
cost1 <- function(v) as.numeric(rastrigin(matrix(v, nrow = 1)))
budget <- res_r$n_eval; rows <- matrix(NA, 10, 4)
for (seed in 1:10) {
  set.seed(seed)
  ga <- do.call(run_ga, c(list(function(P) -rastrigin(P)), ops_real(lo, hi, 2), list(n_pop = 60, n_gen = 80)))
  rs <- min(rastrigin(matrix(runif(budget * 2, lo, hi), ncol = 2)))
  de <- differential_evolution(cost1, lo, hi, 2, NP = 20, maxiter = budget %/% 20 - 1)
  used <- 0; best_gd <- Inf
  while (used < budget) { r <- optim(runif(2, lo, hi), cost1, method = "L-BFGS-B", lower = lo, upper = hi)
    used <- used + r$counts[1] * 3 + 3; best_gd <- min(best_gd, r$value) }
  rows[seed, ] <- c(-ga$best_fitness, rs, de$fun, best_gd)
}
colnames(rows) <- c("genetic algorithm", "random search", "differential evolution", "multi-start gradient descent")
summ <- data.frame(`median final cost` = round(apply(rows, 2, median), 4), `worst of 10` = round(apply(rows, 2, max), 4),
                   `runs with cost < 0.01` = colSums(rows < 0.01), check.names = FALSE)
cat(sprintf("budget per method: about %d fitness evaluations\n", budget)); print(summ)



At equal budget the two evolutionary methods find the global basin in most of the ten runs each, with a median final cost close to zero; the runs that miss converged into one of the basins adjacent to the origin, whose cost is about 0.995 — the "worst of 10" column. Random search with the same number of samples rarely gets below a cost of 0.01: the chance that a uniform sample falls within a few hundredths of the optimum is small, and random search has no mechanism to concentrate. Multi-start gradient descent is the interesting case: each local run converges exactly to the bottom of <em>some</em> basin, so a run either hits zero (if one of its starts fell in the central basin) or stops at about 1. In two dimensions the central basin occupies about 1 % of the square, so a few hundred random starts suffice to hit it, and the advantage of population-based search per unit of effort is modest; Section 7 repeats the comparison in five dimensions, where the picture changes.<sup class="c"><a href="#r21">[21]</a></sup>

### 5.2 A binary problem: the 0/1 knapsack

The <strong>0/1 knapsack problem</strong> is the canonical binary optimisation problem: given \(n\) items with weights \(w_i\) and values \(v_i\) and a capacity \(W\), choose the subset of maximal total value whose total weight does not exceed \(W\). In bioengineering the same structure appears as "which sensors to deploy within a power budget", "which assays to run within a cost", "which features to compute within a latency". The problem is NP-hard in general, but for integer weights it has an exact dynamic-programming solution in \(O(nW)\) time, which the chunk below uses to obtain the true optimum for checking. The chromosome is a mask of \(n\) bits, and the only design decision is what to do with over-weight solutions: the chunk penalises them, so the GA can pass through infeasible regions but is pushed back.<sup class="c"><a href="#r15">[15]</a></sup>



In [ ]:
set.seed(0); n_items <- 30
weights <- sample(5:39, n_items, replace = TRUE); values <- sample(10:99, n_items, replace = TRUE)
capacity <- as.integer(sum(weights) * 0.4)
knapsack_fitness <- function(P, penalty = 5) {
  w <- as.vector(P %*% weights); v <- as.vector(P %*% values)
  ifelse(w <= capacity, v, v - penalty * (w - capacity))
}
knapsack_exact <- function(w, v, W) {                 # DP over capacities: best[c] = max value with weight <= c
  best <- numeric(W + 1)
  for (i in seq_along(w)) { new <- best
    if (w[i] <= W) new[(w[i] + 1):(W + 1)] <- pmax(best[(w[i] + 1):(W + 1)], best[1:(W + 1 - w[i])] + v[i])
    best <- new }
  best[W + 1]
}
optimum <- knapsack_exact(weights, values, capacity)
ga_vals <- rs_vals <- numeric(10)
for (seed in 1:10) {
  set.seed(seed); r <- do.call(run_ga, c(list(knapsack_fitness), ops_binary(n_items), list(n_pop = 80, n_gen = 100)))
  ga_vals[seed] <- if (sum(r$best * weights) <= capacity) sum(r$best * values) else NA
  set.seed(seed); R <- matrix(sample(0:1, r$n_eval * n_items, replace = TRUE), ncol = n_items)
  feas <- as.vector(R %*% weights) <= capacity; rs_vals[seed] <- max(as.vector(R %*% values)[feas])
}
cat(sprintf("%d items, capacity %d (40%% of total weight); exact optimum value = %.0f\n", n_items, capacity, optimum))
cat(sprintf("GA (80 x 100 = %d evals), 10 seeds: optimum found in %d of 10 runs (values %s)\n",
    r$n_eval, sum(ga_vals == optimum, na.rm = TRUE), paste(ga_vals, collapse = " ")))
cat(sprintf("random search, same budget: best feasible values mean %.0f (max %.0f)\n", mean(rs_vals), max(rs_vals)))
cat(sprintf("GA solution (seed 10): %d items, weight %d / %d\n", sum(r$best), sum(r$best * weights), capacity))



The GA reaches the exact optimum (printed above) in most or all runs, with about 8,000 fitness evaluations against a search space of \(2^{30} \approx 10^9\) masks; random search with the same budget stays well below the optimum in every run, because a uniform random mask selects about half the items and is almost always over-weight. When a GA run misses the optimum it typically ends one or two items away, in a solution of nearly the same value, with no indication that anything is missing: a GA offers no certificate of optimality. When an exact method exists at acceptable cost, as here, use it; the GA's domain is the vast majority of problems where none does.

<div class="box key"><h4>Exercise 3</h4><p>Reduce the penalty coefficient from 5 to 0.5 and re-run. The GA now returns infeasible solutions (<code>NA</code> in the list), because exceeding the capacity costs less than the value gained. Then raise it to 50: feasibility is restored, but does the GA still reach the optimum as often? A penalty that is too harsh makes the feasible region's boundary — where the knapsack optimum always lies — hard to approach.</p></div>

### 5.3 A permutation problem: the travelling salesperson

The <strong>travelling-salesperson problem (TSP)</strong> asks for the shortest closed route visiting \(n\) points once each. It is the model problem for every ordering task — the sequence in which a robotic pipette visits wells, a laser scans points, a nurse visits patients, a microscope images regions — and the problem on which permutation operators were developed.<sup class="c"><a href="#r10">[10]</a></sup> The chunk below generates nine random cities, computes the exact optimum by enumerating all \(8!/2 = 20{,}160\) distinct tours (feasible at \(n = 9\) by fixing the first city and permuting the other eight, \(8! = 40{,}320\) orders; hopeless at \(n = 20\), where it would be \(6 \times 10^{16}\)), and runs the permutation GA with order crossover and swap mutation.



In [ ]:
all_perms <- function(x) { if (length(x) == 1) return(matrix(x, 1))
  do.call(rbind, lapply(seq_along(x), function(i) cbind(x[i], all_perms(x[-i])))) }
set.seed(0); n_city <- 9; cities <- matrix(runif(n_city * 2, 0, 100), ncol = 2)
Dm <- as.matrix(dist(cities))
tour_length <- function(P) { P <- rbind(P) + 1; nxt <- cbind(P[, -1, drop = FALSE], P[, 1])
  rowSums(matrix(Dm[cbind(as.vector(P), as.vector(nxt))], nrow = nrow(P))) }
perms <- cbind(0, all_perms(1:(n_city - 1)))          # fix city 0 first
exact <- min(tour_length(perms))
set.seed(0); res_t <- do.call(run_ga, c(list(function(P) -tour_length(P)), ops_permutation(n_city), list(n_pop = 60, n_gen = 60)))
set.seed(1); R <- t(sapply(1:res_t$n_eval, function(i) sample(0:(n_city - 1)))); rs_best <- min(tour_length(R))
succ <- sum(sapply(1:6, function(sd) { set.seed(sd)
  abs(-do.call(run_ga, c(list(function(P) -tour_length(P)), ops_permutation(n_city), list(n_pop = 60, n_gen = 60)))$best_fitness - exact) < 1e-9 }))

par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
route <- c(res_t$best, res_t$best[1]) + 1
plot(cities, pch = 21, bg = ORANGE, col = "black", cex = 1.6, xaxt = "n", yaxt = "n", xlab = "", ylab = "",
     main = "9 cities: GA route (solid) vs best random (dashed)")
lines(cities[route, ], col = BLUE, lwd = 2)
rr <- R[which.min(tour_length(R)), ]; rr <- c(rr, rr[1]) + 1; lines(cities[rr, ], col = GREY, lwd = 1, lty = 2)
text(cities, labels = 0:(n_city - 1), pos = 3, cex = 0.8)
plot(0:60, -res_t$history[, 1], type = "l", col = BLUE, lwd = 2, ylim = range(exact, -res_t$history),
     xlab = "generation", ylab = "tour length", main = "convergence")
lines(0:60, -res_t$history[, 2], col = ORANGE, lwd = 2); abline(h = exact, lty = 3)
legend("topright", c("best in population", "population mean", "exact optimum"),
       col = c(BLUE, ORANGE, "black"), lty = c(1, 1, 3), lwd = 2, bty = "n", cex = 0.8)
cat(sprintf("GA best tour length %.2f; exact optimum %.2f; GA found optimum in %d of 6 seeds; best random tour %.2f\n",
    -res_t$best_fitness, exact, succ, rs_best))



<p class="caption"><b>Figure 11.</b> Left: the nine cities, the GA's best route (solid) and the best of several thousand random permutations (dashed); even the best random tour, at an equal budget, is longer than the GA's optimal route (the two lengths are printed below), and a route that crosses itself can always be shortened by uncrossing, so crossings are a visual sign of sub-optimality. Right: best and mean tour length per generation; the best reaches the exact optimum (dotted) within a few dozen generations, while the mean stays above it, because swap mutation keeps disturbing children every generation — the population retains diversity rather than collapsing onto the best tour. The printed line reports how many of six seeds found the exact optimum.</p>

<div class="box bridge"><h4>Bridge to bioinformatics</h4><p>Multiple sequence alignment, phylogenetic tree search and protein side-chain packing are all combinatorial optimisation problems with no useful gradient, and all three have genetic-algorithm solvers in the literature — SAGA for alignment, GA-based tree search in several phylogenetics packages, and the protein-design work of Jones.<sup class="c"><a href="#r32">[32]</a></sup> The progressive and heuristic methods you used in BGM324 are faster, but they are greedy; a population-based search is the standard way to check whether the greedy answer can be improved.</p></div>

## 6. Why does it work? Schemata, diversity and premature convergence

### 6.1 The schema theorem

Holland's explanation of GA behaviour is built on the notion of a <strong>schema</strong> (plural <em>schemata</em>): a template over the alphabet \(\{0, 1, *\}\) in which \(*\) means "either". The schema `1****` denotes all 5-bit strings beginning with 1 — the integers 16 to 31 — and `1***1` denotes the odd integers above 16. A schema \(H\) has an <em>order</em> \(o(H)\), the number of fixed positions, and a <em>defining length</em> \(\delta(H)\), the distance between its first and last fixed positions: \(o(\text{1****}) = 1\), \(\delta = 0\); \(o(\text{1***1}) = 2\), \(\delta = 4\). Every string of length \(L\) is an instance of \(2^L\) schemata, so a population of \(N\) strings samples many schemata at once — the <em>implicit parallelism</em> Holland regarded as the source of the method's power.<sup class="c"><a href="#r1">[1]</a></sup>

The <strong>schema theorem</strong> states how the number \(m(H, t)\) of instances of a schema changes in one generation of the canonical GA:

<div class="eq">\[ m(H, t+1) \;\ge\; m(H, t)\, \frac{f(H, t)}{\bar f(t)} \left[1 - p_c \frac{\delta(H)}{L - 1} - o(H)\, p_m \right] \]
<span class="where">where \(f(H, t)\) is the mean fitness of the instances of \(H\) and \(\bar f(t)\) the mean fitness of the whole population. The first factor is selection: a schema fitter than average gains copies in proportion. The bracket is a lower bound on the probability that crossover and mutation leave an instance intact: a one-point cut falls inside the schema's defining length with probability \(\delta(H)/(L-1)\), and each of the \(o(H)\) fixed bits survives mutation with probability \(1 - p_m\).</span></div>

The theorem says: <em>short, low-order, above-average schemata receive exponentially increasing numbers of trials in subsequent generations</em>. Goldberg called such schemata <strong>building blocks</strong> and advanced the <strong>building-block hypothesis</strong>: a GA works by discovering, emphasising and recombining building blocks into ever-better complete solutions.<sup class="c"><a href="#r2">[2]</a></sup> The hand example of Section 4 showed exactly this — `11***` and `***11` combining into 11011.

### 6.2 Criticisms

The schema theorem is correct as a statement about one generation, and far weaker than it was once taken to be. Whitley's tutorial and the later literature list the objections:<sup class="c"><a href="#r7">[7]</a></sup>

- It is an <em>inequality</em> about expected values for <em>one</em> step. It says nothing about which schemata are present after many generations, because \(f(H, t)\) changes as the population changes. Iterating the bound as if \(f(H)/\bar f\) were constant — the source of the word "exponentially" — is unjustified.
- It counts the <em>destruction</em> of schemata but ignores their <em>construction</em>: crossover also creates new instances from parents that were not instances. The bound is therefore loose.
- The building-block hypothesis is a hypothesis, not a theorem, and it is false for <em>deceptive</em> problems, where the short, low-order schemata with highest average fitness lead away from the global optimum. Worse, Mitchell, Forrest and Holland designed the <em>royal road</em> functions to be ideal for building-block assembly and found that a simple random-mutation hill-climber outperformed the GA on them; the explanation (<em>hitchhiking</em>) is itself a diversity phenomenon.<sup class="c"><a href="#r4">[4]</a></sup>
- What <em>can</em> be proved rigorously is different in kind: Rudolph showed, by modelling the GA as a Markov chain, that the canonical GA with mutation but without elitism does <em>not</em> converge to the global optimum, while the elitist GA converges with probability one as \(t \to \infty\).<sup class="c"><a href="#r12">[12]</a></sup> This is a guarantee of the weakest kind, but it clarifies what mutation and elitism are for.

The accepted modern position is that the schema theorem is a useful <em>intuition</em>, not a predictive theory. The chunk below tests it on Goldberg's problem with a population of 100: at each generation the lower bound is computed from the current population and compared with the actual count in the next, for the short schema `1****` and the longer `1***1`.



In [ ]:
schema_count <- function(P, schema) {
  chars <- strsplit(schema, "")[[1]]; fixed <- which(chars != "*")
  if (length(fixed) == 0) return(rep(TRUE, nrow(P)))
  vals <- as.integer(chars[fixed]); apply(P[, fixed, drop = FALSE], 1, function(r) all(r == vals))
}
schema_bound <- function(P, f, schema, pc, pm, L = 5) {
  inst <- schema_count(P, schema); m <- sum(inst); if (m == 0) return(0)
  pos <- which(strsplit(schema, "")[[1]] != "*"); o <- length(pos); delta <- max(pos) - min(pos)
  m * mean(f[inst]) / mean(f) * (1 - pc * delta / (L - 1) - o * pm)
}
set.seed(0); N <- 100; pc <- 0.7; pm <- 0.01; n_gen <- 12
P <- matrix(sample(0:1, N * 5, replace = TRUE), nrow = N); rec <- list(`1****` = matrix(NA, 0, 2), `1***1` = matrix(NA, 0, 2))
for (g in 1:n_gen) {
  f <- f_x2(P); bounds <- sapply(names(rec), function(h) schema_bound(P, f, h, pc, pm))
  pool <- P[select_roulette(f, N), ]; pool <- pool[sample(N), ]; kids <- matrix(0, 0, 5)
  for (i in seq(1, N, 2)) { if (runif(1) < pc) { ch <- one_point(pool[i, ], pool[i + 1, ]); kids <- rbind(kids, ch[[1]], ch[[2]]) }
                            else kids <- rbind(kids, pool[i, ], pool[i + 1, ]) }
  P <- mutate_bitflip(kids, pm)
  for (h in names(rec)) rec[[h]] <- rbind(rec[[h]], c(bounds[h], sum(schema_count(P, h))))
}
par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
for (h in names(rec)) { arr <- rec[[h]]
  plot(1:n_gen, arr[, 2], type = "b", pch = 19, col = BLUE, ylim = c(0, N), xlab = "generation t",
       ylab = sprintf("instances in population of %d", N),
       main = sprintf("schema %s: o=%d, delta=%d", h, length(which(strsplit(h, "")[[1]] != "*")),
                      diff(range(which(strsplit(h, "")[[1]] != "*")))))
  lines(1:n_gen, arr[, 1], type = "b", pch = 15, lty = 2, col = ORANGE)
  legend("bottomright", c("actual m(H,t+1)", "schema-theorem bound"), col = c(BLUE, ORANGE), pch = c(19, 15), lty = c(1, 2), bty = "n", cex = 0.75) }
viol <- sapply(names(rec), function(h) sum(rec[[h]][, 2] < rec[[h]][, 1]))
cat(sprintf("generations where actual count fell below the bound: %s (the bound is on the expectation, so occasional violations are sampling noise)\n",
    paste(names(viol), viol, sep = "=", collapse = ", ")))



<p class="caption"><b>Figure 12.</b> Actual number of instances of two schemata in the next generation (blue) and the schema theorem's lower bound from the current generation (orange), on Goldberg's problem with \(N = 100\), \(p_c = 0.7\), \(p_m = 0.01\). Both schemata are above average and both spread quickly: `1****`, whose instances have at least four times the fitness of the rest, is essentially fixed within a couple of generations, so for it the bound is tight. For `1***1` the bound sits well below the actual count: its defining length of 4 makes the crossover-disruption term large — yet the schema is barely disrupted in practice, because crossover between instances <em>creates</em> instances as often as it destroys them. That gap is the second criticism of Section 6.2 made visible.</p>

### 6.3 Exploration versus exploitation

Every stochastic search balances two activities. <strong>Exploration</strong> samples regions about which little is known; <strong>exploitation</strong> refines regions already known to be good. Too much exploration and the method never converges (random search is the limit); too much exploitation and it converges to the first good region it finds (greedy hill-climbing). In a GA, selection pressure and crossover exploit, mutation and a large population explore. The characteristic failure of too much exploitation is <strong>premature convergence</strong>: the population becomes nearly uniform around a sub-optimal solution while the mutation rate is too low to escape.

Premature convergence is detected by measuring <strong>diversity</strong>. Two simple measures for a binary population are the <em>mean pairwise Hamming distance</em> (which equals \(\sum_j 2 p_j (1 - p_j)\) where \(p_j\) is the frequency of allele 1 at locus \(j\)) and the <em>number of distinct individuals</em>. The chunk below runs the binary Rastrigin GA with mild (\(k = 2\)) and strong (\(k = 7\)) selection pressure and tracks both the best cost and the diversity.



In [ ]:
hamming_diversity <- function(P) { p1 <- colMeans(P); sum(2 * p1 * (1 - p1)) }
ga_with_diversity <- function(k, n_gen = 80, seed = 0, pm = 1 / L2, N = 40) {
  set.seed(seed); pop <- matrix(sample(0:1, N * L2, replace = TRUE), nrow = N); fit <- fit_rastrigin_bin(pop); out <- matrix(NA, 0, 3)
  for (g in 1:n_gen) {
    out <- rbind(out, c(-max(fit), hamming_diversity(pop), nrow(unique(pop))))
    elite <- pop[order(fit, decreasing = TRUE)[1], , drop = FALSE]
    par <- pop[select_tournament(fit, 2 * (N - 1), k), , drop = FALSE]; np <- nrow(par) / 2; kids <- matrix(0, np, L2)
    for (i in 1:np) { a <- par[2 * i - 1, ]; b <- par[2 * i, ]; kids[i, ] <- if (runif(1) < 0.8) one_point(a, b)[[1]] else a }
    pop <- rbind(elite, mutate_bitflip(kids, pm)); fit <- fit_rastrigin_bin(pop)
  }
  out
}
avg <- function(k) Reduce(`+`, lapply(0:7, function(sd) ga_with_diversity(k, seed = sd))) / 8
a2 <- avg(2); a7 <- avg(7)
par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
plot(a2[, 1], type = "l", col = BLUE, lwd = 2, log = "y", xlab = "generation", ylab = "best cost (mean of 8 runs)", main = "progress")
lines(a7[, 1], col = ORANGE, lwd = 2)
legend("topright", c(sprintf("k=2: final %.3f", a2[nrow(a2), 1]), sprintf("k=7: final %.3f", a7[nrow(a7), 1])), col = c(BLUE, ORANGE), lwd = 2, bty = "n", cex = 0.8)
plot(a2[, 2], type = "l", col = BLUE, lwd = 2, ylim = c(0, 13), xlab = "generation", ylab = "diversity", main = "diversity of the population")
lines(a7[, 2], col = ORANGE, lwd = 2); lines(a2[, 3] / 40 * 12, col = BLUE, lty = 2); lines(a7[, 3] / 40 * 12, col = ORANGE, lty = 2)
legend("topright", c("k=2 Hamming", "k=7 Hamming", "distinct (scaled)"), col = c(BLUE, ORANGE, "black"), lty = c(1, 1, 2), lwd = c(2, 2, 1), bty = "n", cex = 0.72)



<p class="caption"><b>Figure 13.</b> Left: best cost per generation (mean of eight runs) on the binary-coded Rastrigin problem under mild (\(k = 2\)) and strong (\(k = 7\)) tournament selection. Right: the two diversity measures (the dashed lines are the number of distinct individuals, rescaled to the axis). Strong selection makes slightly faster initial progress — exploitation pays early — but its diversity collapses within about ten generations to the level mutation alone sustains, and the search then crawls; mild selection keeps more distance throughout and ends at a lower cost. The mean Hamming distance of a random population of 24-bit strings is 12; both curves start there and fall, but strong selection falls to a few bits almost immediately. Diversity should be logged in any serious GA run, exactly as the loss is logged in neural-network training (Unit 6): a flat diversity curve near zero means the run is over regardless of the generation counter.</p>

### 6.4 Niching and fitness sharing

Sometimes premature convergence is not a bug to be delayed but a behaviour to be prevented: a multimodal problem may have several optima of comparable quality, and the user wants <em>all</em> of them — alternative treatment plans, alternative molecular scaffolds, alternative feature subsets of equal accuracy. A plain GA converges to one. <strong>Niching</strong> methods make the population settle into several peaks at once. The classic technique is <strong>fitness sharing</strong>, due to Goldberg and Richardson: each individual's fitness is divided by the number of its neighbours within a radius \(\sigma_{\text{share}}\), weighted by proximity,

<div class="eq">\[ f'_i = \frac{f_i}{\sum_j \text{sh}(d_{ij})}, \qquad \text{sh}(d) = \begin{cases} 1 - d/\sigma_{\text{share}} & d < \sigma_{\text{share}} \\ 0 & \text{otherwise} \end{cases} \]
<span class="where">where \(d_{ij}\) is the distance between individuals \(i\) and \(j\). A crowded peak is penalised in proportion to its crowding, so the population redistributes until each peak holds a sub-population proportional to its height.</span></div>

Sareni and Krähenbühl review fitness sharing and the other niching techniques.<sup class="c"><a href="#r13">[13]</a></sup> The chunk below applies sharing to the five-peaked function \(f(x) = \sin^6(5\pi x)\) on \([0, 1]\) that Goldberg and Richardson used, with and without sharing.



In [ ]:
five_peaks <- function(x) sin(5 * pi * x)^6
niching_ga <- function(share, N = 100, n_gen = 60, sigma_share = 0.1, seed = 0) {
  set.seed(seed); x <- runif(N)
  for (g in 1:n_gen) {
    f <- five_peaks(x) + 1e-9
    if (share) { d <- abs(outer(x, x, `-`)); sh <- ifelse(d < sigma_share, 1 - d / sigma_share, 0); f <- f / rowSums(sh) }
    parents <- x[select_sus(f, N)]; parents <- parents[sample(N)]
    kids <- sapply(1:N, function(i) { a <- parents[i]; b <- parents[(i %% N) + 1]; if (runif(1) < 0.7) arithmetic_x(a, b) else a })
    x <- pmin(pmax(kids + rnorm(N, 0, 0.01) * (runif(N) < 0.3), 0), 1)
  }
  x
}
xs1 <- seq(0, 1, length.out = 500); par(mfrow = c(1, 2), mar = c(4, 4, 2.4, 1))
for (share in c(FALSE, TRUE)) {
  xe <- niching_ga(share); counts <- tabulate(pmin(pmax(round(xe * 5 - 0.5), 0), 4) + 1, 5)
  plot(xs1, five_peaks(xs1), type = "l", col = GREY, xlab = "x", ylab = "f(x) = sin^6(5 pi x)",
       main = sprintf("%s fitness sharing: %d of 5 peaks", ifelse(share, "with", "without"), sum(counts > 3)))
  points(xe, five_peaks(xe), pch = 19, col = ifelse(share, ORANGE, BLUE), cex = 0.7)
  cat(sprintf("%s sharing: individuals per peak (0..4): %s\n", ifelse(share, "with   ", "without"), paste(counts, collapse = " ")))
}



<p class="caption"><b>Figure 14.</b> Final population of 100 after 60 generations on a function with five equal peaks. Without sharing (left) the population collapses onto a single peak — the counts per peak are printed — although the five peaks are exactly equal; which peak wins is decided by the early random draws. With fitness sharing (right, \(\sigma_{\text{share}} = 0.1\)) every peak holds a sub-population of comparable size, because an individual on a crowded peak has its fitness divided by the number of its neighbours. The price is that each peak is located less precisely than a single peak would be by a plain GA of the same size.</p>

<div class="box key"><h4>Key idea</h4><p>A GA is a machine for turning diversity into fitness. Selection consumes diversity; mutation, large populations and niching replenish it. Every parameter choice in Section 3 is, in the end, a decision about how fast to spend the diversity — and the only way to know whether the rate was right is to measure it and to run more than one seed.</p></div>

## 7. The wider family: evolutionary and swarm methods

The genetic algorithm is one member of a family the field now calls <strong>evolutionary computation</strong>, together with a neighbouring family of <strong>swarm intelligence</strong> methods and the older physics-inspired <em>simulated annealing</em>. Each paragraph names a method, its characteristic idea and its original reference; Bäck's monograph gives the unified treatment.<sup class="c"><a href="#r6">[6]</a></sup>

**Evolution strategies (ES) and CMA-ES.** Developed by Rechenberg and Schwefel in Berlin in the 1960s for engineering design, evolution strategies work on real-valued vectors, rely on Gaussian mutation as the main operator, and — their decisive idea — <em>adapt the mutation step size during the run</em>: Rechenberg's 1/5 success rule increases \(\sigma\) when more than a fifth of mutations improve and decreases it otherwise. The notation \((\mu, \lambda)\) and \((\mu + \lambda)\) describes whether the \(\mu\) survivors are chosen from the \(\lambda\) children alone or from children and parents together. The modern state of the art is the <strong>covariance matrix adaptation evolution strategy (CMA-ES)</strong> of Hansen and Ostermeier, which adapts the full covariance matrix of the mutation distribution; it is the default for black-box continuous optimisation in up to a few hundred dimensions.<sup class="c"><a href="#r16">[16]</a></sup>

**Genetic programming (GP).** Koza's genetic programming evolves <em>programs</em> rather than parameter vectors: chromosomes are expression trees, crossover swaps sub-trees, and mutation replaces a sub-tree by a random one. Its applications include symbolic regression — discovering the formula, not just the coefficients, that fits data.<sup class="c"><a href="#r17">[17]</a></sup> Its characteristic difficulty is <em>bloat</em>: trees grow without bound unless size is penalised.

**Differential evolution (DE).** Storn and Price's differential evolution, for real-valued problems, generates a candidate by adding to one population member the <em>scaled difference</em> of two others, \(\mathbf{v} = \mathbf{x}_a + F(\mathbf{x}_b - \mathbf{x}_c)\), then recombining and keeping whichever is better. Because the step sizes are drawn from the population's own spread, the method is self-scaling with almost no parameters; it is the method implemented from scratch in Section 5.1 and used again below.<sup class="c"><a href="#r18">[18]</a></sup>

**Particle swarm optimisation (PSO).** Kennedy and Eberhart's particle swarm, inspired by flocking birds, moves a swarm of particles through the search space; each particle has a velocity pulled towards the best position it has personally found and the best the swarm has found. There is no selection, crossover or mutation — information spreads through the social attraction term — and the method is widely used, including for ECG classification with SVMs.<sup class="c"><a href="#r19">[19]</a></sup>

**Simulated annealing (SA).** Kirkpatrick, Gelatt and Vecchi's simulated annealing is a single-point method from statistical physics: a random neighbour is accepted if it is better and, if it is worse by \(\Delta\), with probability \(\exp(-\Delta / T)\), where the temperature \(T\) is lowered slowly. At high temperature it explores, at low temperature it exploits.<sup class="c"><a href="#r20">[20]</a></sup>

**Multi-objective evolution.** Most design problems have several objectives — accuracy and number of features, dose to tumour and dose to healthy tissue, potency and toxicity — that cannot be collapsed into one number without arbitrary weights. Evolutionary algorithms are the dominant approach because a population can represent the whole <em>Pareto front</em> of non-dominated trade-offs at once; Deb's NSGA-II is the reference algorithm.<sup class="c"><a href="#r22">[22]</a></sup>

**The no-free-lunch theorem.** Wolpert and Macready proved that, averaged over <em>all possible</em> objective functions, every optimisation algorithm that never re-evaluates a point has exactly the same performance.<sup class="c"><a href="#r21">[21]</a></sup> The theorem is often misread as "all methods are equally good". What it says is that a method can only outperform others on a <em>class</em> of problems by exploiting structure that the class has — smoothness, separability, the relevance of building blocks. The GA beat random search on Rastrigin, knapsack and TSP because those problems have structure; the theorem guarantees the existence of problems where it would lose. The engineering consequence: the choice of method must be justified by what is known about the problem, not by the method's reputation.

The chunk below implements particle swarm optimisation and a simple simulated annealing in base R and repeats the comparison of Section 5.1 — GA, differential evolution, PSO, simulated annealing and random search — on the Rastrigin function in <em>five</em> dimensions, where the global basin occupies a vanishing fraction of the volume, over six seeds.



In [ ]:
pso <- function(cost, lo, hi, d = 2, n_particles = 30, n_iter = 120, w = 0.7, c1 = 1.5, c2 = 1.5) {
  x <- matrix(runif(n_particles * d, lo, hi), nrow = n_particles); v <- matrix(runif(n_particles * d, -1, 1), nrow = n_particles)
  pbest <- x; pbf <- cost(x); gbest <- pbest[which.min(pbf), ]
  for (it in 1:n_iter) {
    r1 <- matrix(runif(n_particles * d), nrow = n_particles); r2 <- matrix(runif(n_particles * d), nrow = n_particles)
    v <- w * v + c1 * r1 * (pbest - x) + c2 * r2 * (matrix(gbest, n_particles, d, byrow = TRUE) - x)
    x <- pmin(pmax(x + v, lo), hi); f <- cost(x); imp <- f < pbf; pbest[imp, ] <- x[imp, ]; pbf[imp] <- f[imp]
    gbest <- pbest[which.min(pbf), ]
  }
  list(x = gbest, fun = min(pbf), nfev = n_particles * (n_iter + 1))
}
sim_anneal <- function(cost1, lo, hi, d, maxfun = 6000, T0 = 10) {
  x <- runif(d, lo, hi); fx <- cost1(x); best <- x; fbest <- fx
  for (it in 1:maxfun) { Tt <- T0 * (1 - it / maxfun)
    xn <- pmin(pmax(x + rnorm(d, 0, 0.5), lo), hi); fn <- cost1(xn)
    if (fn < fx || runif(1) < exp(-(fn - fx) / max(Tt, 1e-9))) { x <- xn; fx <- fn }
    if (fx < fbest) { best <- x; fbest <- fx } }
  list(x = best, fun = fbest, nfev = maxfun)
}
d5 <- 5; B5 <- 6000; rows5 <- matrix(NA, 6, 5)
rast5 <- function(v) as.numeric(rastrigin(matrix(v, nrow = 1)))
for (seed in 1:6) {
  set.seed(seed)
  ga <- do.call(run_ga, c(list(function(P) -rastrigin(P)), ops_real(lo, hi, d5), list(n_pop = 60, n_gen = B5 %/% 60 - 1)))
  de <- differential_evolution(rast5, lo, hi, d5, NP = 20, maxiter = B5 %/% 20 - 1)
  ps <- pso(function(P) rastrigin(P), lo, hi, d = d5, n_particles = 40, n_iter = B5 %/% 40 - 1)
  sa <- sim_anneal(rast5, lo, hi, d5, maxfun = B5)
  rs <- min(rastrigin(matrix(runif(B5 * d5, lo, hi), ncol = d5)))
  rows5[seed, ] <- c(-ga$best_fitness, de$fun, ps$fun, sa$fun, rs)
}
colnames(rows5) <- c("genetic algorithm", "differential evolution", "particle swarm", "simulated annealing", "random search")
print(data.frame(`median final cost` = round(apply(rows5, 2, median), 3), `best of 6` = round(apply(rows5, 2, min), 3),
           `worst of 6` = round(apply(rows5, 2, max), 3), `runs < 0.01` = colSums(rows5 < 0.01), check.names = FALSE))



In five dimensions the ranking changes and becomes instructive. Random search is now hopeless, because the central basin is a vanishing fraction of the five-dimensional volume. The GA, with its fixed Gaussian step of 0.3, typically ends at a cost of one or two units — each unit of cost is one coordinate stuck in the basin adjacent to zero — because it lacks the fine, adaptive step that would cross the last ridge once the population has shrunk; particle swarm behaves similarly. Differential evolution and simulated annealing do better, because DE's step sizes are drawn from the population's own spread and shrink automatically as the population contracts, and SA's temperature schedule plays the same role. This is the motivation for self-adaptation in evolution strategies and CMA-ES, and the reason a fixed mutation step is the first thing to change in a real-coded GA that stalls. None of this ranks the methods in general — the no-free-lunch theorem forbids that reading — but it shows what each mechanism does and does not provide.<sup class="c"><a href="#r21">[21]</a></sup>

<div class="box key"><h4>Exercise 5</h4><p>Give the GA an adaptive step: in <code>ops_real</code>, replace the fixed <code>sigma</code> by a value that is multiplied by 0.95 every generation (keep a counter in a closure). Re-run the five-dimensional comparison. Then try Rechenberg's 1/5 rule: increase <code>sigma</code> by a factor 1.2 if more than a fifth of the children improved on their parents, decrease it otherwise.</p></div>

## 8. Applications in machine learning and medicine

### 8.1 Feature selection with a GA wrapper

Choosing which features to give a classifier is a binary optimisation problem over \(2^p\) subsets, and it is the application in which GAs entered machine learning. Siedlecki and Sklansky showed in 1989 that a GA with a chromosome of \(p\) bits could select features for a k-NN classifier where exhaustive search is impossible and greedy selection gets trapped;<sup class="c"><a href="#r23">[23]</a></sup> Yang and Honavar made the approach standard, with a fitness combining classifier accuracy and a penalty on the number of features.<sup class="c"><a href="#r24">[24]</a></sup> In the vocabulary of Kohavi and John, this is a <strong>wrapper</strong> method: the learning algorithm is run inside the fitness function, as opposed to a <strong>filter</strong> that ranks features by a statistic computed without the classifier (the \(F\)-test ranking of Figure 1 is a filter).<sup class="c"><a href="#r25">[25]</a></sup> Wrappers are more accurate and far more expensive, and have a specific failure mode this section exhibits.

The experiment uses the breast-cancer data of every unit.<sup class="c"><a href="#r35">[35]</a></sup> The R <code>mlbench</code> version has nine nuclear features, fewer than the thirty of the Python edition's dataset, but the pedagogy is identical: the chromosome is a 9-bit mask; the fitness is the 5-fold cross-validated accuracy of a 5-NN classifier (standardised inside each fold) on the <em>training set only</em>, minus a small penalty per selected feature so that, between two masks of equal accuracy, the smaller is preferred. A cache ensures a mask seen before is not re-evaluated.



In [ ]:
p_feat <- ncol(Xc_tr)                                  # 9 nuclear features in the mlbench version
fs_cache <- new.env()
fs_fitness <- function(P, lam = 0.002) apply(P, 1, function(m) {
  key <- paste(m, collapse = "")
  if (is.null(fs_cache[[key]]))
    fs_cache[[key]] <- if (sum(m) == 0) 0 else cv_knn_acc(Xc_tr[, m == 1, drop = FALSE], yc_tr) - lam * sum(m)
  fs_cache[[key]]
})
set.seed(0); res_fs <- do.call(run_ga, c(list(fs_fitness), ops_binary(p_feat), list(n_pop = 20, n_gen = 20)))
mask_ga <- res_fs$best == 1
plot(0:(nrow(res_fs$history) - 1), res_fs$history[, 1], type = "l", col = BLUE, lwd = 2,
     ylim = range(res_fs$history), xlab = "generation", ylab = "CV accuracy - 0.002 x n_features",
     main = "GA feature selection: fitness per generation")
lines(0:(nrow(res_fs$history) - 1), res_fs$history[, 2], col = ORANGE, lwd = 2)
legend("bottomright", c("best fitness", "mean fitness"), col = c(BLUE, ORANGE), lwd = 2, bty = "n", cex = 0.8)
cat(sprintf("distinct masks evaluated: %d (of 2^%d = %d); best fitness %.4f\n",
    length(ls(fs_cache)), p_feat, 2^p_feat, res_fs$best_fitness))
cat(sprintf("selected %d of %d features: %s\n", sum(mask_ga), p_feat, paste(names(bc)[1:p_feat][mask_ga], collapse = ", ")))



<p class="caption"><b>Figure 15.</b> Best and mean fitness per generation of the feature-selection GA. The best fitness rises in the first generations and then in small steps as the GA trades features for fractions of a percent of cross-validated accuracy; the mean follows below. With only nine features the search space is small, so the GA examines a large fraction of it — the distinct-mask count is printed — but the mechanism is exactly the one that matters when \(p\) is in the dozens or hundreds, where exhaustive search is impossible.</p>

The selected subset has a cross-validated accuracy, on the training set, at least as high as that of all features. That number is <em>not</em> an estimate of how the subset will perform on new patients. The GA examined many masks and returned the one whose cross-validation score was highest; a maximum over many noisy estimates is biased upwards, in exactly the way the training accuracy of a model is biased upwards — the folds were used to <em>choose</em>, so they can no longer <em>evaluate</em>. Ambroise and McLachlan documented this <em>selection bias</em> in gene-expression studies, where selecting genes on the full dataset and then cross-validating produced near-perfect error estimates for subsets with no predictive value.<sup class="c"><a href="#r26">[26]</a></sup> The remedy is a test set that took no part in the selection, or nested cross-validation. The chunk below evaluates the GA subset, the full feature set, a univariate filter of the same size, and the best of 500 random masks of the same size, each on the training folds (optimistic) and on the untouched test set (honest).



In [ ]:
test_knn_acc <- function(mask) {
  sc <- scale(Xc_tr[, mask, drop = FALSE]); ctr <- attr(sc, "scaled:center"); scl <- attr(sc, "scaled:scale"); scl[scl == 0] <- 1
  Xte <- scale(Xc_te[, mask, drop = FALSE], center = ctr, scale = scl); mean(knn(sc, Xte, yc_tr, k = 5) == yc_te)
}
k_sel <- sum(mask_ga)
mask_filter <- rep(FALSE, p_feat); mask_filter[ord_feat[1:k_sel]] <- TRUE
set.seed(0); rand_cv <- numeric(500); rand_masks <- vector("list", 500)
for (i in 1:500) { m <- sample(c(rep(TRUE, k_sel), rep(FALSE, p_feat - k_sel))); rand_masks[[i]] <- m; rand_cv[i] <- cv_knn_acc(Xc_tr[, m, drop = FALSE], yc_tr) }
mask_rand <- rand_masks[[which.max(rand_cv)]]
evaluate <- function(mask) c(cv = cv_knn_acc(Xc_tr[, mask, drop = FALSE], yc_tr), test = test_knn_acc(mask))
tab <- rbind(`all features` = evaluate(rep(TRUE, p_feat)),
             `univariate filter (top k by F)` = evaluate(mask_filter),
             `GA wrapper` = evaluate(mask_ga),
             `best of 500 random k-subsets` = evaluate(mask_rand))
fs_table <- data.frame(`CV acc (used to select)` = round(tab[, 1], 3), `test acc (held out)` = round(tab[, 2], 3),
                       `optimism (CV - test)` = round(tab[, 1] - tab[, 2], 3), check.names = FALSE)
cat(sprintf("median CV accuracy of a random %d-subset: %.3f; best of 500: %.3f; test set has %d cases\n",
    k_sel, median(rand_cv), max(rand_cv), length(yc_te)))
print(fs_table)



The table is the central lesson of this section, read column by column. In the first column — the cross-validation score on the training set — the GA subset has the highest value of the four, as it must, since it was chosen to maximise that column; the best random subset is also above the median random subset (both printed above). In the second column — accuracy on the held-out cases that were never used — the differences shrink: the subset of five features the GA chose generalises as well as the full nine, and all four subsets fall within a few tenths of a percent, which on 205 test cases (one case is about 0.5 %) is within the resolution of the experiment. The third column, the optimism, is largest for the GA wrapper — the subset selected by maximising the fold accuracy over the most candidates — and smallest for the univariate filter, which was chosen without consulting the fold accuracies at all; on this small dataset the full set's optimism is itself large, a reminder that single-split estimates are noisy. Three conclusions survive: a GA wrapper can find a smaller feature set with little loss of accuracy, valuable for interpretability and measurement cost; its reported fitness overstates its accuracy by an amount that grows with the number of candidates examined; and only a held-out set — or nested cross-validation — gives the honest number.<sup class="c"><a href="#r26">[26]</a></sup>

<div class="box warn"><h4>Common misconception</h4><p>"The GA found a subset with 99 % cross-validated accuracy, so our model is 99 % accurate." The first clause is a statement about the <em>search</em>; the second is a claim about <em>new patients</em>, and the cross-validation score cannot support it once it has been used to choose among many candidates. This is the same leakage Supplement S1 warned about for preprocessing, now entering through the optimiser. Any paper that tunes features or hyperparameters by cross-validation and reports that cross-validation score as its result has made this error.</p></div>

<div class="box key"><h4>Exercise 4</h4><p>Change the penalty <code>lam</code> from 0.002 to 0.0 and to 0.01 and re-run both chunks. How many features does the GA keep in each case, and does the test accuracy change? Then change the seed of the GA. The selected subsets will differ while their test accuracies barely differ — the feature-selection landscape has many near-equivalent optima, which is what Section 6.4's niching methods are designed to reveal.</p></div>

### 8.2 Hyperparameter tuning

The hyperparameters of a learning algorithm — \(C\) and \(\gamma\) of an RBF support vector machine (Unit 3), the depth of a tree (Unit 4), the neurons and learning rate of a network (Unit 6) — define a black-box optimisation problem: the objective is a cross-validated score, there is no gradient, evaluations are expensive, and the landscape is often rugged. Grid search is the default; random search is often more efficient when only a few hyperparameters matter; and evolutionary methods are the natural choice when the space is large or mixed, which Friedrichs and Igel demonstrated for SVMs with a CMA-ES.<sup class="c"><a href="#r27">[27]</a></sup> The chunk below tunes \(\log_{10} C \in [-2, 3]\) and \(\log_{10} \gamma \in [-4, 1]\) for an RBF-SVM on the breast-cancer training set with the real-coded GA, and compares with a \(12 \times 12\) grid using about the same number of cross-validations.



In [ ]:
cv_svm_acc <- function(X, y, C, gamma, folds = 5) {
  set.seed(0); fid <- sample(rep(1:folds, length.out = nrow(X)))
  mean(sapply(1:folds, function(fo) { tr <- fid != fo
    m <- svm(X[tr, ], y[tr], cost = C, gamma = gamma, scale = TRUE); mean(predict(m, X[!tr, ]) == y[!tr]) }))
}
svm_cache <- new.env()
svm_cv <- function(theta) { key <- paste(round(theta, 4), collapse = "_")
  if (is.null(svm_cache[[key]])) svm_cache[[key]] <- cv_svm_acc(Xc_tr, yc_tr, 10^theta[1], 10^theta[2])
  svm_cache[[key]] }
svm_fitness <- function(P) apply(P, 1, svm_cv)
box_lo <- c(-2, -4); box_hi <- c(3, 1)
init_svm  <- function(n) cbind(runif(n, box_lo[1], box_hi[1]), runif(n, box_lo[2], box_hi[2]))
cross_svm <- function(a, b) pmin(pmax(blx_alpha(a, b, 0.5), box_lo), box_hi)
mut_svm   <- function(kids) { k <- kids + matrix(rnorm(length(kids), 0, 0.4), nrow = nrow(kids)) * matrix(runif(length(kids)) < 0.3, nrow = nrow(kids))
  k[, 1] <- pmin(pmax(k[, 1], box_lo[1]), box_hi[1]); k[, 2] <- pmin(pmax(k[, 2], box_lo[2]), box_hi[2]); k }
set.seed(0); res_svm <- run_ga(svm_fitness, init_svm, cross_svm, mut_svm, n_pop = 12, n_gen = 12, k = 2)
gc_ax <- seq(-2, 3, length.out = 12); gg_ax <- seq(-4, 1, length.out = 12)
grid_vals <- outer(gc_ax, gg_ax, function(a, b) mapply(function(u, v) svm_cv(c(u, v)), a, b))
bg <- which(grid_vals == max(grid_vals), arr.ind = TRUE)[1, ]; best_grid <- c(gc_ax[bg[1]], gg_ax[bg[2]])
test_of <- function(th) { m <- svm(Xc_tr, yc_tr, cost = 10^th[1], gamma = 10^th[2]); mean(predict(m, Xc_te) == yc_te) }
pts <- do.call(rbind, lapply(ls(svm_cache), function(k) as.numeric(strsplit(k, "_")[[1]])))
image(gc_ax, gg_ax, grid_vals, col = hcl.colors(21, "viridis"), xlab = "log10 C", ylab = "log10 gamma",
      main = "SVM hyperparameter landscape and the GA's search")
contour(gc_ax, gg_ax, grid_vals, add = TRUE, nlevels = 8, col = "white", labcex = 0.6)
points(pts, pch = 21, bg = "white", col = "black", cex = 0.8)
points(res_svm$best[1], res_svm$best[2], pch = 8, col = ORANGE, cex = 2, lwd = 2)
points(best_grid[1], best_grid[2], pch = 22, bg = "red", cex = 1.4)
legend("bottomleft", c("GA points", "GA best", "grid best"), pch = c(21, 8, 22), pt.bg = c("white", NA, "red"),
       col = c("black", ORANGE, "black"), bty = "n", cex = 0.75)
cat(sprintf("GA  : %d CV evaluations, best CV %.4f at log10 C=%.2f, log10 gamma=%.2f -> test accuracy %.3f\n",
    nrow(pts), res_svm$best_fitness, res_svm$best[1], res_svm$best[2], test_of(res_svm$best)))
cat(sprintf("grid: %d CV evaluations, best CV %.4f at log10 C=%.2f, log10 gamma=%.2f -> test accuracy %.3f\n",
    length(grid_vals), max(grid_vals), best_grid[1], best_grid[2], test_of(best_grid)))



<p class="caption"><b>Figure 16.</b> The cross-validated accuracy of the RBF-SVM over the \((\log_{10} C, \log_{10} \gamma)\) plane, from the \(12 \times 12\) grid, with the points the GA evaluated (white), the GA's best (star) and the grid's best (square). The landscape has a broad ridge of near-equal accuracy — large \(C\) compensates small \(\gamma\) — and a region at large \(\gamma\) where the kernel becomes so narrow that every training case is its own island (Unit 3). The GA's points cluster along the ridge after the first generations; both methods reach the same cross-validated accuracy and test accuracies within one test case of each other (printed above).</p>

On a two-dimensional, smooth landscape like this one, grid search is perfectly adequate and the GA has no advantage; the comparison is shown to make the mechanism visible, not to recommend the GA here. The GA's advantages appear when the space has five or ten dimensions (where a grid of 12 points per axis is impossible), when it mixes continuous and categorical choices (kernel type, preprocessing, feature subset — Huang and Wang tuned the SVM's parameters and its feature subset in one chromosome), and when the budget is small. The caveat of Section 8.1 applies unchanged: the best cross-validated score found by either method is optimistic, and the test accuracy is the honest number.

### 8.3 Neuroevolution

Evolution can also replace or complement gradient descent in training neural networks. In <strong>neuroevolution</strong> the chromosome encodes the weights of a network of fixed architecture (a real-valued GA with the loss as negative fitness — useful when the loss is not differentiable, as in reinforcement learning with sparse rewards) or encodes the <em>architecture itself</em>. Stanley and Miikkulainen's NEAT evolves both weights and topology, starting from minimal networks and adding neurons and connections through structural mutations, with a historical-marking scheme that makes crossover between different topologies meaningful and a speciation (niching) mechanism that protects new structures.<sup class="c"><a href="#r28">[28]</a></sup> The modern descendant is <em>neural architecture search</em>, in which evolutionary algorithms compete with reinforcement learning and gradient-based methods to design deep-network architectures; the networks of Unit 8 are the product of such searches more often than of hand design.

### 8.4 Biomedical applications

The evolutionary approach appears in every area of medical bioengineering where a design space is combinatorial and the objective is a simulation or a measured outcome. Five documented examples:

**Radiotherapy treatment planning.** A plan specifies beam directions, shapes and weights so as to deliver a prescribed dose to the tumour while keeping the dose to organs at risk below tolerance — a constrained, multi-objective, non-convex problem. Ezzell applied a GA to the geometric and weight optimisation of three-dimensional conformal plans and found plans competitive with those of experienced planners;<sup class="c"><a href="#r29">[29]</a></sup> multi-objective evolutionary methods of the NSGA-II family now present planners with the trade-off front rather than a single plan.<sup class="c"><a href="#r22">[22]</a></sup>

**De-novo molecular design.** The design space of drug-like molecules is estimated at \(10^{60}\) structures, and the objective — binding, solubility, synthesisability — is computed by models. Jensen's graph-based GA mutates and recombines molecular graphs directly and matched or outperformed deep generative models on standard property-optimisation tasks at a fraction of the cost;<sup class="c"><a href="#r30">[30]</a></sup> in the GuacaMol benchmark suite, a SMILES-based genetic algorithm was among the strongest baselines, ahead of several neural generators.<sup class="c"><a href="#r31">[31]</a></sup> This is the application closest to the QSAR work of BGM324: the GA's fitness function is a QSAR model.

**Protein design.** Jones used a GA to search sequence space for sequences compatible with a given backbone fold, scored by pairwise residue potentials, in one of the first computational de-novo protein design studies;<sup class="c"><a href="#r32">[32]</a></sup> evolutionary search over sequences remains a component of design pipelines that now use deep-learning energy functions.

**Physiological signals.** Feature and parameter selection for ECG arrhythmia classification is a direct instance of Section 8.1: Nasiri and colleagues combined a GA with an SVM to select features and tune the classifier for beat classification on the MIT-BIH database,<sup class="c"><a href="#r33">[33]</a></sup> and Melgani and Bazi did the same with particle swarm optimisation;<sup class="c"><a href="#r19">[19]</a></sup> the same pattern recurs for EEG channel selection and filter-bank design.

**Gene selection in omics data.** With thousands of genes and tens of patients, selecting a small gene signature is the extreme case of the wrapper problem. Li and colleagues' GA/k-NN method selected gene subsets for tumour classification and studied the sensitivity of the result to the GA's parameters, finding many near-equivalent subsets;<sup class="c"><a href="#r34">[34]</a></sup> Ambroise and McLachlan's analysis of selection bias, which Section 8.1 reproduced in miniature, was written about exactly these studies and remains required reading before any omics feature selection.<sup class="c"><a href="#r26">[26]</a></sup>

<div class="box bridge"><h4>Bridge to Unit 9 and to the laboratory</h4><p>Laboratory 5 applies a GA of your own to one of these problem types; the implementation of Section 5 is a starting point, and the checks of Sections 6.3 (log diversity, run several seeds) and 8.1 (hold out a test set) are the criteria it will be assessed on. Unit 9 returns to the ethical dimension: an optimiser is indifferent to what it optimises, and a fitness function that encodes an inequitable label will be optimised with the same efficiency as a fair one.</p></div>

## 9. Summary

You should now be able to explain, without notes:

- Why optimisation is a pillar of AI in its own right, and the four properties of an objective — discrete, non-differentiable, rugged, black-box — under which gradient methods fail and population-based search is appropriate.
- The definition of a genetic algorithm as a population-based stochastic search guided by a fitness function, and the meaning of gene, chromosome, genotype, phenotype, population, generation, fitness, selection, crossover, mutation and elitism.
- The eight design decisions of a GA, with at least two alternatives for each: encoding (binary, Gray, integer, real, permutation); fitness and scaling (raw, linear, rank); selection (roulette, SUS, tournament, rank) and how to measure its pressure (takeover time); crossover (one-point, two-point, uniform; arithmetic, BLX-\(\alpha\), SBX; order crossover); mutation (bit-flip at \(\approx 1/L\), Gaussian, swap); replacement (generational, steady-state) and elitism; termination; and the parameters and their interaction at a fixed budget.
- Goldberg's \(x^2\) example: how to build the fitness table, compute expected counts, spin the roulette wheel, apply one-point crossover at given sites and check for mutation, and why one generation raised the mean fitness from 293 to 439.
- How to write a general GA loop whose problem-specific parts are the fitness function and three operators, and how to apply it to a continuous, a binary and a permutation problem; why random search at equal budget is the minimum baseline.
- The schema theorem — its statement, the meaning of order and defining length, the building-block hypothesis — and its limitations; and what Rudolph's convergence result does and does not guarantee.
- Exploration versus exploitation, premature convergence, how diversity is measured and why it should be logged, and how fitness sharing maintains several niches.
- One paragraph each on evolution strategies and CMA-ES, genetic programming, differential evolution, particle swarm optimisation, simulated annealing and multi-objective evolution, and what the no-free-lunch theorem actually states.
- How a GA wrapper selects features and tunes hyperparameters, why its reported fitness is an optimistically biased estimate, and how a held-out test set or nested cross-validation gives the honest number.
- Five documented biomedical applications: radiotherapy planning, de-novo molecular design, protein design, ECG feature selection and gene-signature selection.

## 10. Self-assessment

<details><summary>1. A colleague wants to use gradient descent to choose which 10 of 40 biomarkers to include in a diagnostic panel. Give two reasons why this cannot work and name an appropriate method.</summary>The search space is discrete (binary masks), so there is no derivative with respect to inclusion; and the objective (cross-validated accuracy) is piecewise constant, so even a relaxed continuous version has zero gradient almost everywhere. A GA wrapper with a 40-bit chromosome (Section 8.1), or another combinatorial search, is appropriate — with a held-out test set for the final estimate.</details>

<details><summary>2. Distinguish genotype and phenotype in a GA that tunes an SVM, and say on which each operator and the fitness function acts.</summary>The genotype is the vector \((\log_{10} C, \log_{10} \gamma)\); the phenotype is the trained SVM with those hyperparameters. Crossover and mutation act on the genotype (they produce new vectors); the fitness function decodes the genotype into a phenotype (fits the SVM) and measures it (cross-validated accuracy).</details>

<details><summary>3. Compute by hand: a population has fitness values 10, 30, 40, 120. What is the selection probability of each under roulette-wheel selection, and the expected number of copies in a mating pool of four?</summary>Sum 200, so probabilities 0.05, 0.15, 0.20, 0.60. Mean fitness 50, so expected copies \(f_i/\bar f\) = 0.2, 0.6, 0.8, 2.4 (which sum to 4). The last individual expects to take over 60 % of the pool in one generation — a super-individual that linear scaling or rank selection would temper.</details>

<details><summary>4. In Goldberg's example, parents 11000 and 10011 crossed at site 2 gave 11011 and 10000. What would crossing at site 3 give, and which site would produce 11111 from these two parents?</summary>Site 3: 110|00 and 100|11 give 11011 and 10000 — the same children, because bit 3 of both parents is 0. No site produces 11111: the parents share a 0 in position 3, and one-point crossover cannot create an allele neither parent carries. Only mutation can supply the missing 1 (Section 3.5).</details>

<details><summary>5. Read the code: in <code>binary_ga</code>, what happens to the best individual if <code>n_elite=0</code>, and which figure shows the consequence?</summary>With <code>n_elite=0</code> the whole next generation consists of mutated offspring of tournament-selected parents; the best survives only if it is selected and then neither crossed over (probability \(1 - p_c\)) nor mutated, so it is frequently lost and the best fitness can fall between generations. Figure 7 shows this, and Rudolph's theorem says the non-elitist GA does not converge to the optimum.</details>

<details><summary>6. Why is a mutation rate of \(1/L\) a sensible default, and what happens at \(p_m = 0.5\)?</summary>At \(p_m = 1/L\) each child differs from its parent by one bit on average, so most children are small variations selection can evaluate meaningfully, while every allele remains reachable. At \(p_m = 0.5\) each bit is set at random regardless of the parent, so children carry no information from their parents and the algorithm degenerates into random search (Figure 6).</details>

<details><summary>7. The schema theorem says above-average, short, low-order schemata receive "exponentially increasing" trials. Why is "exponentially" not justified by the theorem as stated?</summary>The theorem bounds \(m(H, t+1)\) in terms of \(m(H, t)\) and the ratio \(f(H, t)/\bar f(t)\) <em>at generation \(t\)</em>. As the schema spreads, its instances form a larger part of the population and the ratio falls towards 1, so the growth factor is not constant; iterating the one-step bound with a fixed ratio is an unsupported extrapolation. The bound also ignores construction of instances by crossover, so it is loose (Figure 12).</details>

<details><summary>8. Two GA runs stop after 100 generations with the same best fitness. Run A's mean Hamming distance is 0.3 bits, run B's is 6 bits. Which run would benefit more from 100 further generations?</summary>Run B. Run A has converged prematurely — the population is nearly uniform and only mutation can produce new material, slowly; run B still holds diversity that crossover can exploit. Run A would benefit more from a restart with a higher mutation rate or larger population than from continuation (Section 6.3).</details>

<details><summary>9. A GA for feature selection reports a cross-validated accuracy of 0.99 for the selected subset, versus 0.96 for all features. A reviewer says the 0.99 is not an estimate of generalisation performance. Explain the objection and what should be reported instead.</summary>The cross-validation score was the fitness function: the GA examined many subsets and returned the one with the highest score, so the 0.99 is a maximum over many noisy estimates and is biased upwards (selection bias, Ambroise and McLachlan). The accuracy of the selected subset on a test set never used during the search, or a nested cross-validation in which the GA is rerun inside each outer fold, should be reported (Section 8.1).</details>

<details><summary>10. The no-free-lunch theorem states that all optimisers perform equally when averaged over all possible objective functions. Does this make the choice between a GA and simulated annealing arbitrary?</summary>No. The theorem concerns the average over <em>all</em> functions, most of which have no structure. Real problems have structure (smoothness, separability, building blocks, regular basins), and a method performs well on the class whose structure it exploits. The choice must be justified by what is known about the problem and verified empirically against random search at equal budget.</details>

<details><summary>11. Match each method to its idea: CMA-ES, differential evolution, particle swarm optimisation, simulated annealing, genetic programming. Ideas: (a) scaled difference of two members as the mutation step; (b) accept a worse solution with probability \(\exp(-\Delta/T)\); (c) expression trees as chromosomes; (d) adapt the covariance of the mutation distribution; (e) velocity pulled towards personal and global best.</summary>CMA-ES — (d); differential evolution — (a); particle swarm optimisation — (e); simulated annealing — (b); genetic programming — (c).</details>

<details><summary>12. In a radiotherapy planning GA, the fitness is "dose to tumour minus dose to spinal cord". A plan scores very highly by delivering an enormous dose to the tumour and a dose to the cord just under tolerance. What has happened, and what does it illustrate about fitness design?</summary>The GA has optimised exactly the function it was given: without an upper bound on tumour dose or a constraint on other organs, the linear fitness rewards unlimited dose. The algorithm did nothing wrong — the fitness did not express what was wanted. Constraints must be encoded (penalties, bounds, or a multi-objective formulation), and the result must be inspected by a human before use (Sections 3.2, 7 and 8.4).</details>

## Glossary

<dl class="gloss">
<dt>Allele</dt><dd>The value held by a gene at a given position (locus) of a chromosome; 0 or 1 in a binary GA.</dd>
<dt>Building block</dt><dd>A short, low-order schema with above-average fitness; the building-block hypothesis holds that a GA assembles solutions from such schemata by crossover.</dd>
<dt>Chromosome (individual)</dt><dd>The complete encoded representation of one candidate solution — a bit string, a real vector or a permutation.</dd>
<dt>Crossover (recombination)</dt><dd>An operator that produces offspring by combining genes of two parents; one-point, two-point and uniform for strings, arithmetic, BLX-\(\alpha\) and SBX for real vectors, order crossover for permutations.</dd>
<dt>Defining length δ(H)</dt><dd>The distance between the first and last fixed positions of a schema; long schemata are more likely to be cut by one-point crossover.</dd>
<dt>Differential evolution</dt><dd>A real-valued evolutionary algorithm whose mutation step is the scaled difference of two population members.</dd>
<dt>Diversity</dt><dd>A measure of how different the individuals of a population are (mean pairwise Hamming distance, number of distinct individuals, spread around the centroid); its collapse signals premature convergence.</dd>
<dt>Elitism</dt><dd>Copying the best one or few individuals unchanged into the next generation, so the best fitness never decreases.</dd>
<dt>Encoding (representation)</dt><dd>The mapping between candidate solutions and chromosomes; the decoding function maps genotype to phenotype.</dd>
<dt>Evolution strategy</dt><dd>A real-valued evolutionary algorithm relying on Gaussian mutation with self-adapted step sizes; CMA-ES adapts the full covariance matrix.</dd>
<dt>Exploration / exploitation</dt><dd>Sampling unknown regions / refining regions already known to be good; mutation and large populations explore, selection and crossover exploit.</dd>
<dt>Fitness function</dt><dd>The objective function, to be maximised, evaluated on the phenotype; the only information a GA uses about the problem.</dd>
<dt>Fitness scaling</dt><dd>A transformation of raw fitness (linear scaling, ranking) that controls selection pressure under fitness-proportional selection.</dd>
<dt>Generation</dt><dd>One iteration of the GA loop: selection, crossover, mutation, replacement.</dd>
<dt>Genetic programming</dt><dd>Evolutionary search over programs or expressions represented as trees.</dd>
<dt>Genotype / phenotype</dt><dd>The encoded chromosome that the operators act on / the decoded solution that the fitness evaluates.</dd>
<dt>Mutation</dt><dd>Random perturbation of a chromosome (bit-flip, Gaussian, swap); the source of alleles not present in the population and the guarantee that every point is reachable.</dd>
<dt>Niching (fitness sharing)</dt><dd>Techniques that let a population occupy several optima at once by penalising crowded regions.</dd>
<dt>No-free-lunch theorem</dt><dd>Averaged over all possible objective functions, all optimisers perform identically; superiority on a problem class comes from exploiting its structure.</dd>
<dt>Order (of a schema) o(H)</dt><dd>The number of fixed (non-*) positions in a schema.</dd>
<dt>Particle swarm optimisation</dt><dd>A swarm method in which particles move with velocities attracted to their personal best and the swarm's best position.</dd>
<dt>Premature convergence</dt><dd>Loss of population diversity around a sub-optimal solution before the optimum has been found.</dd>
<dt>Roulette-wheel selection</dt><dd>Fitness-proportional selection: individual \(i\) is chosen with probability \(f_i / \sum_j f_j\).</dd>
<dt>Schema</dt><dd>A template over {0, 1, *} denoting a set of chromosomes; the unit of analysis of Holland's schema theorem.</dd>
<dt>Selection pressure</dt><dd>The degree to which selection favours fitter individuals; measured by takeover time or the expected copies of the best individual.</dd>
<dt>Simulated annealing</dt><dd>A single-point search that accepts worse neighbours with probability \(\exp(-\Delta/T)\) under a decreasing temperature \(T\).</dd>
<dt>Stochastic universal sampling</dt><dd>Fitness-proportional selection with one spin and \(N\) equally spaced pointers, minimising sampling error.</dd>
<dt>Tournament selection</dt><dd>Choosing the best of \(k\) randomly drawn individuals; \(k\) sets the selection pressure.</dd>
<dt>Wrapper / filter</dt><dd>Feature selection that evaluates subsets by running the classifier / by a statistic computed without the classifier.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Holland JH. <em>Adaptation in Natural and Artificial Systems</em>. Ann Arbor: University of Michigan Press; 1975. Reissued Cambridge, MA: MIT Press; 1992. <a href="https://doi.org/10.7551/mitpress/1090.001.0001">doi:10.7551/mitpress/1090.001.0001</a></li>
<li id="r2">Goldberg DE. <em>Genetic Algorithms in Search, Optimization and Machine Learning</em>. Reading, MA: Addison-Wesley; 1989. ISBN 978-0-201-15767-3. <a href="https://dl.acm.org/doi/book/10.5555/534133">dl.acm.org/doi/book/10.5555/534133</a> (no DOI; ACM record)</li>
<li id="r3">De Jong KA. <em>An Analysis of the Behavior of a Class of Genetic Adaptive Systems</em>. PhD thesis, University of Michigan; 1975. <a href="https://hdl.handle.net/2027.42/4507">hdl.handle.net/2027.42/4507</a> (no DOI)</li>
<li id="r4">Mitchell M. <em>An Introduction to Genetic Algorithms</em>. Cambridge, MA: MIT Press; 1996. <a href="https://doi.org/10.7551/mitpress/3927.001.0001">doi:10.7551/mitpress/3927.001.0001</a></li>
<li id="r5">Eiben AE, Smith JE. <em>Introduction to Evolutionary Computing</em>. 2nd ed. Berlin: Springer; 2015. <a href="https://doi.org/10.1007/978-3-662-44874-8">doi:10.1007/978-3-662-44874-8</a></li>
<li id="r6">Bäck T. <em>Evolutionary Algorithms in Theory and Practice</em>. New York: Oxford University Press; 1996. <a href="https://doi.org/10.1093/oso/9780195099713.001.0001">doi:10.1093/oso/9780195099713.001.0001</a></li>
<li id="r7">Whitley D. A genetic algorithm tutorial. <em>Stat Comput</em>. 1994;4(2):65–85. <a href="https://doi.org/10.1007/BF00175354">doi:10.1007/BF00175354</a></li>
<li id="r8">Blickle T, Thiele L. A comparison of selection schemes used in evolutionary algorithms. <em>Evol Comput</em>. 1996;4(4):361–394. <a href="https://doi.org/10.1162/evco.1996.4.4.361">doi:10.1162/evco.1996.4.4.361</a></li>
<li id="r9">Eshelman LJ, Schaffer JD. Real-coded genetic algorithms and interval-schemata. In: <em>Foundations of Genetic Algorithms 2</em>. Morgan Kaufmann; 1993:187–202. <a href="https://doi.org/10.1016/B978-0-08-094832-4.50018-0">doi:10.1016/B978-0-08-094832-4.50018-0</a></li>
<li id="r10">Larrañaga P, Kuijpers CMH, Murga RH, Inza I, Dizdarevic S. Genetic algorithms for the travelling salesman problem: a review of representations and operators. <em>Artif Intell Rev</em>. 1999;13(2):129–170. <a href="https://doi.org/10.1023/A:1006529012972">doi:10.1023/A:1006529012972</a></li>
<li id="r11">Srinivas M, Patnaik LM. Adaptive probabilities of crossover and mutation in genetic algorithms. <em>IEEE Trans Syst Man Cybern</em>. 1994;24(4):656–667. <a href="https://doi.org/10.1109/21.286385">doi:10.1109/21.286385</a></li>
<li id="r12">Rudolph G. Convergence analysis of canonical genetic algorithms. <em>IEEE Trans Neural Netw</em>. 1994;5(1):96–101. <a href="https://doi.org/10.1109/72.265964">doi:10.1109/72.265964</a></li>
<li id="r13">Sareni B, Krähenbühl L. Fitness sharing and niching methods revisited. <em>IEEE Trans Evol Comput</em>. 1998;2(3):97–106. <a href="https://doi.org/10.1109/4235.735432">doi:10.1109/4235.735432</a></li>
<li id="r14">Mühlenbein H, Schomisch M, Born J. The parallel genetic algorithm as function optimizer. <em>Parallel Comput</em>. 1991;17(6–7):619–632. <a href="https://doi.org/10.1016/S0167-8191(05)80052-3">doi:10.1016/S0167-8191(05)80052-3</a></li>
<li id="r15">Khuri S, Bäck T, Heitkötter J. The zero/one multiple knapsack problem and genetic algorithms. In: <em>Proc. 1994 ACM Symposium on Applied Computing</em>. 1994:188–193. <a href="https://doi.org/10.1145/326619.326694">doi:10.1145/326619.326694</a></li>
<li id="r16">Hansen N, Ostermeier A. Completely derandomized self-adaptation in evolution strategies. <em>Evol Comput</em>. 2001;9(2):159–195. <a href="https://doi.org/10.1162/106365601750190398">doi:10.1162/106365601750190398</a></li>
<li id="r17">Koza JR. Genetic programming as a means for programming computers by natural selection. <em>Stat Comput</em>. 1994;4(2):87–112. <a href="https://doi.org/10.1007/BF00175355">doi:10.1007/BF00175355</a></li>
<li id="r18">Storn R, Price K. Differential evolution — a simple and efficient heuristic for global optimization over continuous spaces. <em>J Glob Optim</em>. 1997;11(4):341–359. <a href="https://doi.org/10.1023/A:1008202821328">doi:10.1023/A:1008202821328</a></li>
<li id="r19">Kennedy J, Eberhart R. Particle swarm optimization. In: <em>Proc. ICNN'95</em>. 1995;4:1942–1948. <a href="https://doi.org/10.1109/ICNN.1995.488968">doi:10.1109/ICNN.1995.488968</a>. ECG application: Melgani F, Bazi Y. Classification of electrocardiogram signals with support vector machines and particle swarm optimization. <em>IEEE Trans Inf Technol Biomed</em>. 2008;12(5):667–677. <a href="https://doi.org/10.1109/TITB.2008.923147">doi:10.1109/TITB.2008.923147</a></li>
<li id="r20">Kirkpatrick S, Gelatt CD, Vecchi MP. Optimization by simulated annealing. <em>Science</em>. 1983;220(4598):671–680. <a href="https://doi.org/10.1126/science.220.4598.671">doi:10.1126/science.220.4598.671</a></li>
<li id="r21">Wolpert DH, Macready WG. No free lunch theorems for optimization. <em>IEEE Trans Evol Comput</em>. 1997;1(1):67–82. <a href="https://doi.org/10.1109/4235.585893">doi:10.1109/4235.585893</a></li>
<li id="r22">Deb K, Pratap A, Agarwal S, Meyarivan T. A fast and elitist multiobjective genetic algorithm: NSGA-II. <em>IEEE Trans Evol Comput</em>. 2002;6(2):182–197. <a href="https://doi.org/10.1109/4235.996017">doi:10.1109/4235.996017</a></li>
<li id="r23">Siedlecki W, Sklansky J. A note on genetic algorithms for large-scale feature selection. <em>Pattern Recognit Lett</em>. 1989;10(5):335–347. <a href="https://doi.org/10.1016/0167-8655(89)90037-8">doi:10.1016/0167-8655(89)90037-8</a></li>
<li id="r24">Yang J, Honavar V. Feature subset selection using a genetic algorithm. <em>IEEE Intell Syst</em>. 1998;13(2):44–49. <a href="https://doi.org/10.1109/5254.671091">doi:10.1109/5254.671091</a></li>
<li id="r25">Kohavi R, John GH. Wrappers for feature subset selection. <em>Artif Intell</em>. 1997;97(1–2):273–324. <a href="https://doi.org/10.1016/S0004-3702(97)00043-X">doi:10.1016/S0004-3702(97)00043-X</a></li>
<li id="r26">Ambroise C, McLachlan GJ. Selection bias in gene extraction on the basis of microarray gene-expression data. <em>Proc Natl Acad Sci USA</em>. 2002;99(10):6562–6566. <a href="https://doi.org/10.1073/pnas.102102699">doi:10.1073/pnas.102102699</a></li>
<li id="r27">Friedrichs F, Igel C. Evolutionary tuning of multiple SVM parameters. <em>Neurocomputing</em>. 2005;64:107–117. <a href="https://doi.org/10.1016/j.neucom.2004.11.022">doi:10.1016/j.neucom.2004.11.022</a></li>
<li id="r28">Stanley KO, Miikkulainen R. Evolving neural networks through augmenting topologies. <em>Evol Comput</em>. 2002;10(2):99–127. <a href="https://doi.org/10.1162/106365602320169811">doi:10.1162/106365602320169811</a></li>
<li id="r29">Ezzell GA. Genetic and geometric optimization of three-dimensional radiation therapy treatment planning. <em>Med Phys</em>. 1996;23(3):293–305. <a href="https://doi.org/10.1118/1.597660">doi:10.1118/1.597660</a></li>
<li id="r30">Jensen JH. A graph-based genetic algorithm and generative model/Monte Carlo tree search for the exploration of chemical space. <em>Chem Sci</em>. 2019;10(12):3567–3572. <a href="https://doi.org/10.1039/C8SC05372C">doi:10.1039/C8SC05372C</a></li>
<li id="r31">Brown N, Fiscato M, Segler MHS, Vaucher AC. GuacaMol: benchmarking models for de novo molecular design. <em>J Chem Inf Model</em>. 2019;59(3):1096–1108. <a href="https://doi.org/10.1021/acs.jcim.8b00839">doi:10.1021/acs.jcim.8b00839</a></li>
<li id="r32">Jones DT. De novo protein design using pairwise potentials and a genetic algorithm. <em>Protein Sci</em>. 1994;3(4):567–574. <a href="https://doi.org/10.1002/pro.5560030405">doi:10.1002/pro.5560030405</a></li>
<li id="r33">Nasiri JA, Naghibzadeh M, Yazdi HS, Naghibzadeh B. ECG arrhythmia classification with support vector machines and genetic algorithm. In: <em>2009 Third UKSim European Symposium on Computer Modeling and Simulation</em>. IEEE; 2009:187–192. <a href="https://doi.org/10.1109/EMS.2009.39">doi:10.1109/EMS.2009.39</a></li>
<li id="r34">Li L, Weinberg CR, Darden TA, Pedersen LG. Gene selection for sample classification based on gene expression data: study of sensitivity to choice of parameters of the GA/KNN method. <em>Bioinformatics</em>. 2001;17(12):1131–1142. <a href="https://doi.org/10.1093/bioinformatics/17.12.1131">doi:10.1093/bioinformatics/17.12.1131</a></li>
<li id="r35">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r36">R Core Team. <em>R: A Language and Environment for Statistical Computing</em>. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; no DOI)</li>
<li id="r37">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package. <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
<li id="r38">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. New York: Springer; 2002. <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a> (the reference for the R modelling functions <code>knn</code> and <code>svm</code> used here)</li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4th ed. Pearson; 2021. — Chapter 4 covers hill climbing, simulated annealing, genetic algorithms and local search in continuous spaces.</li>
<li>Eiben AE, Smith JE. <em>Introduction to Evolutionary Computing</em>. 2nd ed. Springer; 2015. — The standard textbook; the eight components of Section 3 follow its organisation.</li>
<li>Goldberg DE. <em>Genetic Algorithms in Search, Optimization and Machine Learning</em>. Addison-Wesley; 1989. — Chapter 1 contains the hand-worked example of Section 4; Chapter 2 the schema theorem.</li>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — The R companion to the whole course; free PDF at statlearning.com.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Section 7.10.2 ("The wrong and right way to do cross-validation") formalises the selection bias of Section 8.1.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>CRAN Task View: Optimization and Mathematical Programming — <a href="https://cran.r-project.org/view=Optimization">cran.r-project.org/view=Optimization</a>; R packages for global optimisation, including the <code>GA</code> and <code>DEoptim</code> packages, useful once the mechanics of this notebook are understood (not used here, which implements the GA from scratch).</li>
<li>Nikolaus Hansen's CMA-ES page — <a href="http://cma-es.github.io">cma-es.github.io</a>; tutorial and reference implementations.</li>
<li>GuacaMol benchmark — <a href="https://github.com/BenevolentAI/guacamol">github.com/BenevolentAI/guacamol</a>; the de-novo molecular design benchmark of Section 8.4, with the graph-based GA among its baselines.</li>
<li>UCI Machine Learning Repository — <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a>; original source of the breast-cancer data.</li>
</ul>
